# Pilot Machine Learning Model for Fuel Requirement Estimation in Agricultural Mechanization

## Authors

Marcus Rafael B. Tiongson

Ronald Melvin R. Rosas

## Table of Contents

1. [Introduction](#Introduction)
2. [About the Dataset](#About-the-Dataset)
   - [Dataset Setup](#1.-Dataset-Setup)
3. [Data Pre-processing](#Data-Pre-processing)
   - [1. Setup](#1.-Setup)
     - [1A. Imported Libraries](#1A.-Imported-Libraries)
     - [1B. Configuration](#1B.-Configuration)
   - [2. Data Ingestion](#2.-Data-Ingestion)
     - [2A. ABEMIS ZIP Extractor](#2A.-ABEMIS-ZIP-Extractor)
     - [2B. AMTEC PDF Extractor V4](#2B.-AMTEC-PDF-Extractor-V4)
   - [3. Data Quality & Classification](#3.-Data-Quality-&-Classification)
     - [3A. ABEMIS Usability Check](#3A.-ABEMIS-Usability-Check)
     - [3B. ABEMIS Fuel-Relevance Separator](#3B.-ABEMIS-Fuel-Relevance-Separator)
     - [3C. AMTEC Analytics & Cleaning](#3C.-AMTEC-Analytics-&-Cleaning)
     - [3D. AMTEC Regression Modeling](#3D.-AMTEC-Regression-Modeling)
   - [Results Summary - Data Pre-processing](#6.-Results-Summary---Data-Pre-processing)
4. [Machine Learning Implementation](#Machine-Learning-Implementation)
   - [Setup: Baseline Model (OLS)](#Setup:-Baseline-Model)
     - [1A. OLS Coefficient Table](#1A.-OLS-Coefficient-Table)
     - [1B. OLS Diagnostic Plots](#1B.-OLS-Diagnostic-Plots)
   - [Setup: Random Forest Regressor](#Setup:-Random-Forest-Regressor)
     - [2A. SHAP Feature Importance](#2A.-SHAP-Feature-Importance)
     - [2B. LIME Local Explanations](#2B.-LIME-Local-Explanations)
   - [Improvements](#Improvements)
     - [ABEMIS Context Features](#ABEMIS-Context-Features)
     - [OCR Retraining](#OCR-Retraining)
   - [Results Summary - Machine Learning Implementation](#Results-Summary---Machine-Learning-Implementation)
   - [Analysis of Results](#Analysis-of-Results)
5. [Conclusion](#Conclusion)
6. [Recommendation](#Recommendation)

# Introduction

This notebook documents an end-to-end pilot pipeline for estimating fuel requirements of Philippine agricultural machinery from two official datasets:

* **AMTEC**: Test reports (performance data, PDFs)
* **ABEMIS**: Machinery inventory (regional distribution, ZIP/Excel)

**Pipeline stages:**
* Ingest raw sources
* Clean and join into 376-record training set
* Fit OLS baseline (hierarchical: GLOBAL → FAMILY → TYPE)
* Train Random Forest regressor on 8 features
* Explain predictions globally with SHAP and locally with LIME
* Apply trained model to 246,741 ABEMIS records for national scoring

## About the Dataset


## 1. Dataset Setup
The dataset will consist of structured records of agricultural machinery operations. Each row will represent one operation instance, such as one machine performing one operation over a defined area and time period.

| Variable Group            | Variables                                                                 | Purpose                                                     |
|---------------------------|---------------------------------------------------------------------------|-------------------------------------------------------------|
| Identification            | Record ID, date, region, province, municipality                            | Traceability, geographic grouping, and data cleaning        |
| Machine characteristics   | Machine type, horsepower, fuel type, machine age                           | Represents equipment-related fuel demand                   |
| Operation characteristics | Operation type, implement used (if available), number of passes (if available) | Represents the type and intensity of field operation        |
| Operational effort        | Area covered, operating time, travel distance                              | Normalizes fuel use per hectare or per hour                |
| Field/location indicators | Crop type, soil type, terrain, irrigation/rainfed status (if available)   | Captures contextual variability                            |
| Target variable           | Fuel consumption (liters)                                                  | Output variable for model training and evaluation           |
| Derived variables         | Fuel per hour, fuel per hectare, time per hectare                          | Supports exploratory analysis and model interpretation      |

**Datasets**

* **Agricultural Machinery Testing and Evaluation Center (AMTEC)** (https://bit.ly/AMTECTestReports2026)
  - Fuel consumption and performance data
  
* **Agricultural and Biosystems Engineering Management Information System (ABEMIS)** (https://abemis.bafe.gov.ph/user/login)
  - Machinery inventory and distribution data
  
* **Regional Agricultural Engineering Division (RAED) Cropping Calendar**
  - Crop stage and seasonal information

# Data Pre-processing

## 1. Setup

### 1A. Imported Libraries

**Key library choices:**

* **PyMuPDF (`fitz`)**: Fastest pure-Python PDF text reader
  - Handles AMTEC's text-native reports without external binaries
  
* **`statsmodels`**: Used for OLS regression
  - Returns full diagnostics (t-stats, p-values, leverage, Cook's D)
  - `sklearn.LinearRegression` only returns coefficients
  
* **`scikit-learn`**: Random Forest baseline + train/test split utilities

* **Tesseract bridge** (`pytesseract` + `pdf2image`): Conditional loading
  - For small subset of AMTEC PDFs that are scanned images
  - Text-native reports bypass OCR entirely

* **`lime`** for LIME

In [ ]:
!pip install pymupdf pandas openpyxl numpy matplotlib scipy statsmodels scikit-learn lime

# Uncomment if USE_OCR = True in config:
!pip install pytesseract pdf2image pillow

In [ ]:
# Standard library
from pathlib import Path
import zipfile
import shutil
import re
import warnings
from datetime import datetime

# Data
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt

# PDF extraction
import fitz  # PyMuPDF

# Statistics / Regression
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import OLSInfluence
from scipy import stats

# Machine learning
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

warnings.filterwarnings("ignore")

### 1B. Configuration

**Purpose:** Every downstream cell (ingestion, analytics, modeling) consumes the same set of input/output directories.

**Benefits:**
* Single location to update when project moves machines
* Keeps individual cells focused on logic rather than path plumbing
* In the refactored module tree, same constants live in `config.py`

> **Update all paths in the cell below before running the notebook.**

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive', force_remount=True)

In [ ]:
# ============================================================
# CONFIGURATION  --  edit BASE_DIR below if your data lives elsewhere
# ============================================================
#
# This cell is the SINGLE source of truth for every path and modeling
# constant used downstream. Run it once at notebook start. Every other
# cell reads from these globals; no module imports required.
#
# Expected on-disk layout:
#   <DATA_ROOT>/Agricultural Machinery Inventory from ABEMIS/
#     Analytics Output V2/AMTEC_Test_Report_Fuel_Power_Analytics_V2.xlsx   <-- ML training set
#     Regression Parameters Output V3/                                      <-- model outputs land here
#     Extracted Batches Improved V3/                                        <-- AMTEC PDF extraction outputs
#
# The notebook ships inside Mini-Project/Project Documents/ alongside the
# paper. If you place this notebook somewhere else, set DATA_ROOT manually.

from pathlib import Path

# Auto-detect: search several candidates relative to the notebook's working
# directory for the expected folder structure. Override DATA_ROOT manually if
# auto-detection picks the wrong location.
_cwd = Path.cwd()
_marker = "Agricultural Machinery Inventory from ABEMIS"
_candidates = [
    _cwd,                              # notebook sits inside Mini-Project/
    _cwd.parent,                       # notebook sits inside Mini-Project/<subdir>/  (default for this repo)
    _cwd / "Mini-Project",             # notebook sits in the parent of Mini-Project
    _cwd.parent / "Mini-Project",      # one level up + Mini-Project
]
DATA_ROOT = next((p for p in _candidates if (p / _marker).exists()), _cwd)
BASE_DIR  = DATA_ROOT.parent  # for backward compatibility with older cells

# If auto-detection failed, set this manually:
# DATA_ROOT = Path('/absolute/path/to/Mini-Project')

# ABEMIS directories
ABEMIS_RAW_DIR       = DATA_ROOT / "Agricultural Machinery Inventory from ABEMIS/Raw"
ABEMIS_EXTRACTED_DIR = DATA_ROOT / "Agricultural Machinery Inventory from ABEMIS/Extracted"
ABEMIS_DIAG_DIR      = DATA_ROOT / "Agricultural Machinery Inventory from ABEMIS/Diagnostics"
ABEMIS_FUEL_DIR      = DATA_ROOT / "Agricultural Machinery Inventory from ABEMIS/Fuel Classification V2"

# AMTEC directories  --  outputs are co-located inside the ABEMIS folder
AMTEC_PDF_DIR        = DATA_ROOT / "Test Reports from BAFE-AMTEC"
AMTEC_EXTRACTION_DIR = DATA_ROOT / "Agricultural Machinery Inventory from ABEMIS/Extracted Batches Improved V3"
AMTEC_ANALYTICS_DIR  = DATA_ROOT / "Agricultural Machinery Inventory from ABEMIS/Analytics Output V2"
AMTEC_REGRESSION_DIR = DATA_ROOT / "Agricultural Machinery Inventory from ABEMIS/Regression Parameters Output V3"

# OCR settings (only used if pytesseract + pdf2image are installed)
USE_OCR       = True
OCR_MAX_PAGES = 3

# Regression / RF settings
ALPHA                              = 0.05
MIN_RECORDS                        = 5
MIN_UNIQUE_POWER                   = 3
MIN_ACCEPTABLE_R2                  = 0.50
R2_FILTER_TYPE                     = "adjusted"   # "adjusted" or "raw"
EXCLUDE_EXTREME_OUTLIERS_FOR_FINAL = True
CLIP_NEGATIVE_PREDICTIONS          = True

# Random Forest feature set + reproducibility constants
TARGET_COL = "fuel_l_per_hr"
NUMERIC_FEATURES = [
    "power_kw", "year",
    "abemis_total_count", "abemis_region_breadth", "abemis_dominant_region_share",
]
CATEGORICAL_FEATURES = ["machinery_type", "machinery_family", "analysis_subset"]
FEATURE_COLS = NUMERIC_FEATURES + [f"{c}_code" for c in CATEGORICAL_FEATURES]
TEST_SIZE    = 0.2
RANDOM_STATE = 42

# Create output directories
for _d in [ABEMIS_EXTRACTED_DIR, ABEMIS_DIAG_DIR, ABEMIS_FUEL_DIR,
           AMTEC_EXTRACTION_DIR, AMTEC_ANALYTICS_DIR, AMTEC_REGRESSION_DIR]:
    _d.mkdir(parents=True, exist_ok=True)

print("Configuration loaded. Output directories ready.")
print(f"  Notebook cwd : {_cwd}")
print(f"  DATA_ROOT    : {DATA_ROOT}")
print(f"  Training xlsx: {AMTEC_ANALYTICS_DIR / 'AMTEC_Test_Report_Fuel_Power_Analytics_V2.xlsx'}")
print(f"  Outputs land : {AMTEC_REGRESSION_DIR}")


## 2. Data Ingestion

**Why two ingestion stages?**

* Project combines two officially-distributed datasets in incompatible formats:
  - ABEMIS ships region-labelled ZIP archives of Excel files
  - AMTEC publishes individual PDF test reports
  
* Each stage isolates failure modes:
  - Corrupted PDF doesn't block ABEMIS pipeline
  - Corrupted ABEMIS file doesn't block AMTEC extraction

### 2A. ABEMIS ZIP Extractor

Extracts region-labeled ZIP files into organised Excel files per region.

ABEMIS distributes the inventory as numerically-suffixed ZIP archives (`machineries_inv (3).zip`) where the suffix encodes the region but isn't meaningful on its own. Mapping suffix → canonical region code (`NCR`, `CAR`, `R1`...) at extraction time means every downstream join can key on the region name directly instead of fragile file-system artefacts.

In [ ]:
TEMP_DIR = ABEMIS_EXTRACTED_DIR / "_temp_extracted"

TEMP_DIR.mkdir(parents=True, exist_ok=True)

region_map = {
    "machineries_inv": "NCR",
    "machineries_inv (1)": "CAR",
    "machineries_inv (2)": "R1",
    "machineries_inv (3)": "R2",
    "machineries_inv (4)": "R3",
    "machineries_inv (5)": "R4A",
    "machineries_inv (6)": "R4B",
    "machineries_inv (7)": "R5",
    "machineries_inv (8)": "R6",
    "machineries_inv (9)": "R7",
    "machineries_inv (10)": "R8",
    "machineries_inv (11)": "R9",
    "machineries_inv (12)": "R10",
    "machineries_inv (13)": "R11",
    "machineries_inv (14)": "R12",
    "machineries_inv (15)": "R13",
    "machineries_inv (16)": "BARMM",
    "machineries_inv (17)": "NIRR",
}

def get_batch_no(name):
    match = re.search(r"output_batch[_\s-]*(\d+)", name, re.IGNORECASE)
    return int(match.group(1)) + 1 if match else None

def normalize_stem(path):
    return path.stem.strip()

total_copied = 0

zip_files = list(ABEMIS_RAW_DIR.rglob("*.zip"))

print("ZIP files found:", len(zip_files))

for zip_path in zip_files:
    zip_stem = normalize_stem(zip_path)

    if zip_stem not in region_map:
        print("Skipped unmapped ZIP:", zip_path.name)
        continue

    region = region_map[zip_stem]
    extract_to = TEMP_DIR / zip_stem
    extract_to.mkdir(parents=True, exist_ok=True)

    print(f"\nExtracting {zip_path.name} -> {region}")

    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(extract_to)

    batch_files = [
        p for p in extract_to.rglob("*")
        if p.is_file() and "output_batch" in p.name.lower()
    ]

    print("Batch files found:", len(batch_files))

    for file in batch_files:
        batch_no = get_batch_no(file.name)

        if batch_no is None:
            print("Skipped, no batch number:", file.name)
            continue

        ext = file.suffix if file.suffix else ".csv"
        destination = ABEMIS_EXTRACTED_DIR / f"{region} batch {batch_no}{ext}"

        counter = 2
        while destination.exists():
            destination = ABEMIS_EXTRACTED_DIR / f"{region} batch {batch_no} copy {counter}{ext}"
            counter += 1

        shutil.copy2(file, destination)
        total_copied += 1

        print("Copied:", file.name, "->", destination.name)

print("\nDONE")
print("Total copied:", total_copied)
print("Output folder:", ABEMIS_EXTRACTED_DIR)

### 2B. AMTEC PDF Extractor V4

Parses AMTEC test report PDFs to extract machinery type, rated power, and fuel consumption.

**Why two extraction paths?**

* **Text-native PDFs**: PyMuPDF reads in milliseconds
* **Scanned images**: Tesseract OCR (slower, recovers ~24 additional records)
* `MIN_TEXT_LENGTH` threshold decides which path each PDF takes

**Why batch every 100 PDFs?**

* Full corpus extraction takes ~30 minutes
* Single crash mid-run would lose all progress
* Checkpoints every 100 PDFs for fault tolerance

In [ ]:
# Local settings (global settings from config above)
MIN_TEXT_LENGTH = 300
CHECKPOINT_EVERY = 100

# ============================================================
# AMTEC TEST REPORT FULL EXTRACTOR - V4 NO BATCHING
# ============================================================
#

# ============================================================
# SETTINGS
# ============================================================

MIN_TEXT_LENGTH = 300

CHECKPOINT_EVERY = 100   # autosave every 100 PDFs

# ============================================================
# MACHINERY DICTIONARY
# ============================================================

MACHINE_PATTERNS = {
    "Solar-Powered Irrigation System": [
        r"\bSPIS\b",
        r"solar[-\s]?powered irrigation system",
    ],

    "Walking-Type Agricultural Tractor": [
        r"walking[-\s]?type agricultural tractor",
        r"walk[-\s]?behind agricultural tractor",
    ],

    "Four-Wheel Tractor": [
        r"four[-\s]?wheel tractor",
        r"4[-\s]?wheel tractor",
        r"\b4wt\b",
    ],

    "Hand Tractor": [
        r"hand tractor",
        r"walking[-\s]?type tractor",
        r"two[-\s]?wheel tractor",
        r"2[-\s]?wheel tractor",
        r"\bpower tiller\b",
    ],

    "Small Engine": [
        r"small engine",
        r"gasoline engine",
        r"diesel engine",
    ],

    "Rotary Tiller": [
        r"rotary tiller",
        r"rotavator",
        r"rotary cultivator",
    ],

    "Combine Harvester": [
        r"combine harvester",
        r"rice combine",
        r"corn combine",
    ],

    "Rice Transplanter": [
        r"rice transplanter",
        r"mechanical rice transplanter",
        r"walk[-\s]?behind transplanter",
        r"riding[-\s]?type transplanter",
    ],

    "Reaper": [
        r"\breaper\b",
        r"rice reaper",
    ],

    "Water Pump": [
        r"water pump",
        r"irrigation pump",
        r"\bpump\b",
    ],

    "Mechanical Dryer": [
        r"mechanical dryer",
        r"flatbed dryer",
        r"recirculating dryer",
        r"grain dryer",
        r"batch dryer",
    ],

    "Thresher": [
        r"\bthresher\b",
        r"rice thresher",
    ],

    "Sheller": [
        r"\bsheller\b",
        r"corn sheller",
    ],

    "Rice Mill": [
        r"rice mill",
        r"milling machine",
        r"rice milling",
    ],

    "Seeder": [
        r"\bseeder\b",
        r"seed drill",
    ],

    "Sprayer": [
        r"\bsprayer\b",
        r"power sprayer",
    ],
}

LOW_PRIORITY_PATTERNS = [
    r"moisture meter",
    r"tester",
    r"analyzer",
    r"weighing scale",
    r"laboratory",
    r"meter",
    r"sensor",
]

# ============================================================
# TEXT FUNCTIONS
# ============================================================

def clean_text(text):
    if text is None:
        return ""

    text = str(text)
    text = text.replace("\xa0", " ")
    text = text.replace("ﬁ", "fi").replace("ﬂ", "fl")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n+", "\n", text)
    return text.strip()

def extract_text_pymupdf(pdf_path):
    try:
        doc = fitz.open(pdf_path)
        text = ""

        for page in doc:
            text += page.get_text("text") + "\n"

        doc.close()
        text = clean_text(text)

        if len(text) < MIN_TEXT_LENGTH:
            return text, "LOW_TEXT_POSSIBLE_SCANNED"

        return text, "TEXT_EXTRACTED"

    except Exception as e:
        return "", f"PDF_READ_ERROR: {e}"

def extract_text_ocr(pdf_path, max_pages=3):
    try:
        import pytesseract
        from pdf2image import convert_from_path

        pages = convert_from_path(
            str(pdf_path),
            first_page=1,
            last_page=max_pages,
            dpi=200
        )

        text = ""
        for img in pages:
            text += pytesseract.image_to_string(img, lang="eng") + "\n"

        text = clean_text(text)

        if len(text) < MIN_TEXT_LENGTH:
            return text, "OCR_LOW_TEXT"

        return text, "OCR_EXTRACTED"

    except Exception as e:
        return "", f"OCR_ERROR: {e}"

def extract_text(pdf_path):
    text, status = extract_text_pymupdf(pdf_path)

    if USE_OCR and len(text) < MIN_TEXT_LENGTH:
        ocr_text, ocr_status = extract_text_ocr(pdf_path, OCR_MAX_PAGES)

        if len(ocr_text) > len(text):
            return ocr_text, ocr_status

    return text, status

# ============================================================
# HELPER FUNCTIONS
# ============================================================

def clean_field_value(value):
    if value is None:
        return None

    value = str(value)
    value = value.replace("\n", " ")
    value = re.sub(r"\s+", " ", value)
    value = value.strip(" :;,.|-–")
    return value.strip()

def is_valid_field_value(value):
    if value is None:
        return False

    value = str(value).strip()

    if len(value) == 0:
        return False

    if re.fullmatch(r"[,.\s/-]+", value):
        return False

    return True

def find_first(patterns, text, flags=re.IGNORECASE):
    for pattern in patterns:
        match = re.search(pattern, text, flags)
        if match:
            value = clean_field_value(match.group(1))
            if is_valid_field_value(value):
                return value

    return None

def extract_number_and_unit(value):
    if value is None:
        return None, None

    value = str(value).replace(",", "")

    match = re.search(
        r"([0-9]+(?:\.[0-9]+)?)\s*([A-Za-z/%\-\^0-9]+(?:/[A-Za-z0-9]+)?)?",
        value
    )

    if not match:
        return None, None

    return float(match.group(1)), match.group(2)

def kw_to_hp(kw):
    return kw * 1.34102209 if kw is not None else None

def hp_to_kw(hp):
    return hp * 0.745699872 if hp is not None else None

# ============================================================
# EXTRACTION FUNCTIONS
# ============================================================

def extract_report_no(text, filename):
    combined = f"{filename}\n{text}"

    value = find_first([
        r"Test\s*Report\s*No\.?\s*[:\-]?\s*((?:19|20)\d{2}[-–]\d{3,5})",
        r"Report\s*No\.?\s*[:\-]?\s*((?:19|20)\d{2}[-–]\d{3,5})",
        r"\b((?:19|20)\d{2}[-–]\d{3,5})\b",
    ], combined)

    return value.replace("–", "-") if value else None

def extract_machinery_type(text, filename):
    combined = f"{filename}\n{text}".lower()
    scores = {}

    for machine_type, patterns in MACHINE_PATTERNS.items():
        score = 0

        for pattern in patterns:
            hits = re.findall(pattern, combined, flags=re.IGNORECASE)
            score += len(hits)

            if re.search(pattern, filename.lower(), flags=re.IGNORECASE):
                score += 5

        if score > 0:
            scores[machine_type] = score

    if not scores:
        return None

    return max(scores, key=scores.get)

def extract_from_filename(pdf_path):
    stem = pdf_path.stem

    report_no = None
    m = re.search(r"\b((?:19|20)\d{2}[-–]\d{3,5})\b", stem)
    if m:
        report_no = m.group(1).replace("–", "-")

    cleaned = re.sub(r"\b(?:19|20)\d{2}[-–]\d{3,5}\b", "", stem)
    cleaned = cleaned.replace("_", " ").replace("-", " ")
    cleaned = re.sub(r"\s+", " ", cleaned).strip()

    machinery_type = extract_machinery_type("", cleaned)

    brand = None
    model = None
    temp = cleaned

    if machinery_type:
        for pattern in MACHINE_PATTERNS.get(machinery_type, []):
            temp = re.sub(pattern, "", temp, flags=re.IGNORECASE).strip()

    temp = re.sub(r"\s+", " ", temp).strip(" -_")
    parts = temp.split()

    if len(parts) >= 1:
        brand = parts[0]

    if len(parts) >= 2:
        model = " ".join(parts[1:])

    return report_no, machinery_type, brand, model

def classify_project_relevance(text, filename, machinery_type):
    combined = f"{filename}\n{text}".lower()

    if machinery_type in MACHINE_PATTERNS:
        return "HIGH_RELEVANCE"

    for pattern in LOW_PRIORITY_PATTERNS:
        if re.search(pattern, combined):
            return "LOW_RELEVANCE"

    return "UNKNOWN_RELEVANCE"

def extract_matched_keywords(text, filename):
    combined = f"{filename}\n{text}".lower()
    matched = []

    for machine_type, patterns in MACHINE_PATTERNS.items():
        for pattern in patterns:
            if re.search(pattern, combined):
                matched.append(machine_type)
                break

    return ", ".join(sorted(set(matched)))

def extract_brand(text):
    return find_first([
        r"\bBrand\s*[:\-]\s*([A-Za-z0-9][A-Za-z0-9 \-/().,&]{1,60})",
        r"\bTrade\s*Name\s*[:\-]\s*([A-Za-z0-9][A-Za-z0-9 \-/().,&]{1,60})",
        r"\bMake\s*[:\-]\s*([A-Za-z0-9][A-Za-z0-9 \-/().,&]{1,60})",
    ], text)

def extract_model(text):
    return find_first([
        r"\bModel\s*No\.?\s*[:\-]\s*([A-Za-z0-9][A-Za-z0-9 \-/().,&]{1,60})",
        r"\bModel\s*[:\-]\s*([A-Za-z0-9][A-Za-z0-9 \-/().,&]{1,60})",
    ], text)

def extract_power_raw(text):
    return find_first([
        r"Rated\s*Power\s*[:\-]?\s*([0-9]+(?:\.[0-9]+)?\s*(?:kW|KW|kw|hp|HP|Hp))",
        r"Engine\s*Power\s*[:\-]?\s*([0-9]+(?:\.[0-9]+)?\s*(?:kW|KW|kw|hp|HP|Hp))",
        r"Power\s*Rating\s*[:\-]?\s*([0-9]+(?:\.[0-9]+)?\s*(?:kW|KW|kw|hp|HP|Hp))",
        r"Maximum\s*Power\s*[:\-]?\s*([0-9]+(?:\.[0-9]+)?\s*(?:kW|KW|kw|hp|HP|Hp))",
        r"\b([0-9]+(?:\.[0-9]+)?\s*kW)\b",
        r"\b([0-9]+(?:\.[0-9]+)?\s*hp)\b",
    ], text)

def normalize_power(power_raw):
    value, unit = extract_number_and_unit(power_raw)

    if value is None or unit is None:
        return None, None

    unit_l = unit.lower()

    if "kw" in unit_l:
        power_kw = value
        power_hp = kw_to_hp(value)
    elif "hp" in unit_l:
        power_hp = value
        power_kw = hp_to_kw(value)
    else:
        return None, None

    if power_kw <= 0 or power_kw > 500:
        return None, None

    return round(power_kw, 3), round(power_hp, 3)

def extract_fuel_raw(text):
    return find_first([
        r"Fuel\s*Consumption\s*[:\-]?\s*([0-9]+(?:\.[0-9]+)?\s*(?:L/hr|L/h|l/hr|l/h|li/hr|L/ha|l/ha|kg/hr|kg/h))",
        r"Specific\s*Fuel\s*Consumption\s*[:\-]?\s*([0-9]+(?:\.[0-9]+)?\s*(?:L/hr|L/h|l/hr|l/h|g/kWh|g/hp-hr))",
        r"Fuel\s*Rate\s*[:\-]?\s*([0-9]+(?:\.[0-9]+)?\s*(?:L/hr|L/h|l/hr|l/h))",
        r"Diesel\s*Consumption\s*[:\-]?\s*([0-9]+(?:\.[0-9]+)?\s*(?:L/hr|L/h|l/hr|l/h|L/ha|l/ha))",
        r"Gasoline\s*Consumption\s*[:\-]?\s*([0-9]+(?:\.[0-9]+)?\s*(?:L/hr|L/h|l/hr|l/h|L/ha|l/ha))",
        r"\b([0-9]+(?:\.[0-9]+)?\s*(?:L/hr|L/h|l/hr|l/h|li/hr|L/ha|l/ha))\b",
    ], text)

def normalize_fuel(fuel_raw):
    value, unit = extract_number_and_unit(fuel_raw)

    if value is None:
        return None, None

    if value <= 0 or value > 300:
        return None, None

    if unit:
        unit = unit.replace("l", "L")
        unit = unit.replace("L/hr", "L/h")
        unit = unit.replace("L/H", "L/h")

    return value, unit

def extract_field_capacity_raw(text):
    return find_first([
        r"Effective\s*Field\s*Capacity\s*[:\-]?\s*([0-9]+(?:\.[0-9]+)?\s*(?:ha/hr|ha/h|hectare/hr|hectares/hr))",
        r"Theoretical\s*Field\s*Capacity\s*[:\-]?\s*([0-9]+(?:\.[0-9]+)?\s*(?:ha/hr|ha/h|hectare/hr|hectares/hr))",
        r"Field\s*Capacity\s*[:\-]?\s*([0-9]+(?:\.[0-9]+)?\s*(?:ha/hr|ha/h|hectare/hr|hectares/hr))",
        r"\b([0-9]+(?:\.[0-9]+)?\s*(?:ha/hr|ha/h))\b",
    ], text)

def extract_operating_speed_raw(text):
    return find_first([
        r"Operating\s*Speed\s*[:\-]?\s*([0-9]+(?:\.[0-9]+)?\s*(?:km/hr|km/h|kph))",
        r"Forward\s*Speed\s*[:\-]?\s*([0-9]+(?:\.[0-9]+)?\s*(?:km/hr|km/h|kph))",
        r"Travel\s*Speed\s*[:\-]?\s*([0-9]+(?:\.[0-9]+)?\s*(?:km/hr|km/h|kph))",
        r"\b([0-9]+(?:\.[0-9]+)?\s*(?:km/hr|km/h|kph))\b",
    ], text)

def validate_speed(speed_raw):
    value, unit = extract_number_and_unit(speed_raw)

    if value is None:
        return None, None

    if value <= 0 or value > 40:
        return None, None

    return value, unit

def extract_general_capacity_raw(text):
    return find_first([
        r"Output\s*Capacity\s*[:\-]?\s*([0-9]+(?:\.[0-9]+)?\s*(?:kg/hr|kg/h|tons/hr|ton/hr|t/hr|cavans/hr|bags/hr))",
        r"Throughput\s*Capacity\s*[:\-]?\s*([0-9]+(?:\.[0-9]+)?\s*(?:kg/hr|kg/h|tons/hr|ton/hr|t/hr))",
        r"Capacity\s*[:\-]?\s*([0-9]+(?:\.[0-9]+)?\s*(?:kg/hr|kg/h|tons/hr|ton/hr|t/hr|cavans/hr|bags/hr))",
    ], text)

def extract_fuel_type(text):
    value = find_first([
        r"Fuel\s*Type\s*[:\-]?\s*(Diesel|Gasoline|Petrol|Kerosene)",
        r"\b(Diesel|Gasoline|Petrol|Kerosene)\b",
    ], text)

    return value.title() if value else None

# ============================================================
# RECORD EXTRACTION
# ============================================================

def extract_record(pdf_path):
    text, status = extract_text(pdf_path)

    filename_report_no, filename_machine, filename_brand, filename_model = extract_from_filename(pdf_path)

    report_no = extract_report_no(text, pdf_path.name) or filename_report_no
    year = report_no[:4] if report_no else None

    machinery_type = extract_machinery_type(text, pdf_path.name) or filename_machine

    brand = extract_brand(text) or filename_brand
    model = extract_model(text) or filename_model

    if machinery_type == "Solar-Powered Irrigation System":
        brand = None
        model = None

    relevance = classify_project_relevance(text, pdf_path.name, machinery_type)
    matched_keywords = extract_matched_keywords(text, pdf_path.name)

    power_raw = extract_power_raw(text)
    power_kw, power_hp = normalize_power(power_raw)

    fuel_raw = extract_fuel_raw(text)
    fuel_value, fuel_unit = normalize_fuel(fuel_raw)

    field_capacity_raw = extract_field_capacity_raw(text)
    field_capacity_value, field_capacity_unit = extract_number_and_unit(field_capacity_raw)

    speed_raw = extract_operating_speed_raw(text)
    speed_value, speed_unit = validate_speed(speed_raw)

    general_capacity_raw = extract_general_capacity_raw(text)
    general_capacity_value, general_capacity_unit = extract_number_and_unit(general_capacity_raw)

    return {
        "test_report_no": report_no,
        "year": year,
        "machinery_type": machinery_type,
        "brand": brand,
        "model": model,

        "rated_power_raw": power_raw,
        "power_kw": power_kw,
        "power_hp": power_hp,

        "fuel_type": extract_fuel_type(text),
        "fuel_consumption_raw": fuel_raw,
        "fuel_value": fuel_value,
        "fuel_unit": fuel_unit,

        "field_capacity_raw": field_capacity_raw,
        "field_capacity_value": field_capacity_value,
        "field_capacity_unit": field_capacity_unit,

        "operating_speed_raw": speed_raw,
        "operating_speed_value": speed_value,
        "operating_speed_unit": speed_unit,

        "general_capacity_raw": general_capacity_raw,
        "general_capacity_value": general_capacity_value,
        "general_capacity_unit": general_capacity_unit,

        "project_relevance": relevance,
        "matched_keywords": matched_keywords,

        "source_file": pdf_path.name,
        "source_path": str(pdf_path),
        "extraction_status": status,
        "text_length": len(text),
        "needs_ocr": len(text) < MIN_TEXT_LENGTH,
    }

# ============================================================
# FULL PROCESS
# ============================================================

all_pdfs = sorted(AMTEC_PDF_DIR.rglob("*.pdf"))
total_files = len(all_pdfs)

print("Total PDFs found:", total_files)
print("OCR enabled:", USE_OCR)
print("Output folder:", AMTEC_EXTRACTION_DIR)

records = []

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

for i, pdf_path in enumerate(all_pdfs, start=1):
    print(f"[{i}/{total_files}] {pdf_path.name}")

    try:
        records.append(extract_record(pdf_path))

    except Exception as e:
        records.append({
            "test_report_no": None,
            "year": None,
            "machinery_type": None,
            "brand": None,
            "model": None,
            "rated_power_raw": None,
            "power_kw": None,
            "power_hp": None,
            "fuel_type": None,
            "fuel_consumption_raw": None,
            "fuel_value": None,
            "fuel_unit": None,
            "field_capacity_raw": None,
            "field_capacity_value": None,
            "field_capacity_unit": None,
            "operating_speed_raw": None,
            "operating_speed_value": None,
            "operating_speed_unit": None,
            "general_capacity_raw": None,
            "general_capacity_value": None,
            "general_capacity_unit": None,
            "project_relevance": "ERROR",
            "matched_keywords": "",
            "source_file": pdf_path.name,
            "source_path": str(pdf_path),
            "extraction_status": f"ERROR: {e}",
            "text_length": 0,
            "needs_ocr": True,
        })

    if i % CHECKPOINT_EVERY == 0:
        checkpoint_df = pd.DataFrame(records)
        checkpoint_file = AMTEC_EXTRACTION_DIR / f"AMTEC_full_extraction_checkpoint_{i}_{timestamp}.xlsx"
        checkpoint_df.to_excel(checkpoint_file, index=False)
        print("Checkpoint saved:", checkpoint_file)

# ============================================================
# FINAL DATAFRAME
# ============================================================

df = pd.DataFrame(records)

df["missing_machinery_type"] = df["machinery_type"].isna()
df["missing_fuel_consumption"] = df["fuel_value"].isna()
df["missing_rated_power"] = df["power_kw"].isna()
df["missing_field_capacity"] = df["field_capacity_value"].isna()

df["usable_for_core_dataset"] = (
    (df["project_relevance"] == "HIGH_RELEVANCE") &
    df["machinery_type"].notna() &
    df["fuel_value"].notna() &
    df["power_kw"].notna()
)

df["high_priority_for_review"] = (
    (df["project_relevance"] == "HIGH_RELEVANCE") &
    ~df["usable_for_core_dataset"]
)

# ============================================================
# SAVE OUTPUTS
# ============================================================

full_file = AMTEC_EXTRACTION_DIR / "AMTEC_full_extracted_dataset.xlsx"
core_file = AMTEC_EXTRACTION_DIR / "AMTEC_core_usable_dataset.xlsx"
review_file = AMTEC_EXTRACTION_DIR / "AMTEC_high_priority_for_review.xlsx"
ocr_file = AMTEC_EXTRACTION_DIR / "AMTEC_needs_ocr.xlsx"
summary_file = AMTEC_EXTRACTION_DIR / "AMTEC_full_extraction_summary.txt"

df.to_excel(full_file, index=False)

df[df["usable_for_core_dataset"] == True].to_excel(core_file, index=False)

df[df["high_priority_for_review"] == True].to_excel(review_file, index=False)

df[df["needs_ocr"] == True].to_excel(ocr_file, index=False)

summary = f"""
AMTEC FULL EXTRACTION SUMMARY

Total PDFs processed: {len(df)}

Text extracted: {(df['extraction_status'] == 'TEXT_EXTRACTED').sum()}
Low text / possible scanned: {(df['extraction_status'] == 'LOW_TEXT_POSSIBLE_SCANNED').sum()}
OCR extracted: {(df['extraction_status'] == 'OCR_EXTRACTED').sum()}
OCR low text: {(df['extraction_status'] == 'OCR_LOW_TEXT').sum()}

High relevance: {(df['project_relevance'] == 'HIGH_RELEVANCE').sum()}
Low relevance: {(df['project_relevance'] == 'LOW_RELEVANCE').sum()}
Unknown relevance: {(df['project_relevance'] == 'UNKNOWN_RELEVANCE').sum()}

Machinery type extracted: {df['machinery_type'].notna().sum()}
Rated power extracted: {df['power_kw'].notna().sum()}
Fuel consumption extracted: {df['fuel_value'].notna().sum()}
Field capacity extracted: {df['field_capacity_value'].notna().sum()}
Operating speed extracted: {df['operating_speed_value'].notna().sum()}

Needs OCR: {df['needs_ocr'].sum()}
Usable for core dataset: {df['usable_for_core_dataset'].sum()}
High priority for review: {df['high_priority_for_review'].sum()}

Output files:
1. {full_file}
2. {core_file}
3. {review_file}
4. {ocr_file}
"""

with open(summary_file, "w", encoding="utf-8") as f:
    f.write(summary)

print(summary)
print("DONE.")

## 3. Data Quality & Classification

Raw extraction outputs are heterogeneous:

* Different ABEMIS regions report different columns
* AMTEC PDFs use mixed units
* Many ABEMIS rows describe non-fuel machinery (electric mills, food processors)

**Goal:** Classify and standardize everything **before** modeling so:
* Training set is clean
* Inference target is well-defined

### 3A. ABEMIS Usability Check

Scans extracted ABEMIS files and reports which key column categories are present.

Some regional ABEMIS files lack rated-power or quantity columns entirely. Knowing which regions are usable *before* running the classifier prevents silent loss of records and lets us flag regions that need manual cleanup separately.

In [ ]:
# ============================================================
# ABEMIS INVENTORY DATA USABILITY CHECKER
# ============================================================

# ============================================================
# FOLDER LOCATIONS
# ============================================================

OUTPUT_FILE = ABEMIS_DIAG_DIR / "ABEMIS_Inventory_Usability_Check.xlsx"

# ============================================================
# COLUMN KEYWORDS
# ============================================================

MACHINERY_KEYWORDS = [
    "machine", "machinery", "equipment", "implement", "commodity",
    "type", "classification", "category"
]

LOCATION_KEYWORDS = [
    "region", "province", "municipality", "city", "barangay",
    "location", "psgc"
]

BRAND_KEYWORDS = [
    "brand", "make", "manufacturer"
]

MODEL_KEYWORDS = [
    "model", "model no", "model number"
]

POWER_KEYWORDS = [
    "hp", "horsepower", "kw", "power", "rated"
]

QUANTITY_KEYWORDS = [
    "quantity", "qty", "count", "number", "units", "no.", "total"
]

OWNER_KEYWORDS = [
    "owner", "beneficiary", "farmer", "fca", "association", "cooperative"
]

DATE_KEYWORDS = [
    "date", "year", "acquired", "encoded", "created", "updated"
]

# ============================================================
# HELPER FUNCTIONS
# ============================================================

def normalize_colname(col):
    col = str(col).strip().lower()
    col = re.sub(r"[^a-z0-9]+", "_", col)
    col = re.sub(r"_+", "_", col)
    return col.strip("_")

def find_matching_columns(columns, keywords):
    matches = []

    for col in columns:
        col_norm = normalize_colname(col)

        for key in keywords:
            key_norm = normalize_colname(key)

            if key_norm in col_norm:
                matches.append(col)
                break

    return matches

def infer_region_from_filename(file_path):
    name = file_path.stem

    match = re.search(r"(NCR|CAR|BARMM|NIRR|R\d+[A-Z]?)", name, re.IGNORECASE)
    if match:
        return match.group(1).upper()

    return None

def read_file_safely(file_path):
    suffix = file_path.suffix.lower()

    try:
        if suffix in [".xlsx", ".xls"]:
            df = pd.read_excel(file_path)

        elif suffix == ".csv":
            try:
                df = pd.read_csv(file_path, encoding="utf-8")
            except UnicodeDecodeError:
                df = pd.read_csv(file_path, encoding="latin1")

        elif suffix == ".json":
            df = pd.read_json(file_path)

        else:
            return None, f"Unsupported file type: {suffix}"

        return df, "OK"

    except Exception as e:
        return None, f"Read error: {e}"

def missing_rate(series):
    return series.isna().mean()

def classify_usability(row_count, col_count, machinery_cols, location_cols):
    if row_count == 0:
        return "UNUSABLE - No rows"

    if col_count == 0:
        return "UNUSABLE - No columns"

    if len(machinery_cols) == 0 and len(location_cols) == 0:
        return "LOW - Cannot detect machinery or location columns"

    if len(machinery_cols) == 0:
        return "PARTIAL - Location only, machinery column missing"

    if len(location_cols) == 0:
        return "PARTIAL - Machinery only, location columns missing"

    return "GOOD - Has machinery and location indicators"

# ============================================================
# MAIN PROCESS
# ============================================================

all_files = []

for ext in ["*.xlsx", "*.xls", "*.csv", "*.json"]:
    all_files.extend(ABEMIS_EXTRACTED_DIR.rglob(ext))

all_files = sorted(all_files)

print("Files found:", len(all_files))

file_summaries = []
column_summaries = []
sample_rows = []

for i, file_path in enumerate(all_files, start=1):
    print(f"[{i}/{len(all_files)}] Checking: {file_path.name}")

    df, status = read_file_safely(file_path)

    inferred_region = infer_region_from_filename(file_path)

    if df is None:
        file_summaries.append({
            "file_name": file_path.name,
            "file_path": str(file_path),
            "inferred_region": inferred_region,
            "read_status": status,
            "row_count": 0,
            "column_count": 0,
            "duplicate_rows": None,
            "machinery_columns": None,
            "location_columns": None,
            "brand_columns": None,
            "model_columns": None,
            "power_columns": None,
            "quantity_columns": None,
            "owner_columns": None,
            "date_columns": None,
            "usability_rating": "UNUSABLE - Read error"
        })
        continue

    # Drop fully blank rows and columns for diagnostics only
    df = df.dropna(how="all")
    df = df.dropna(axis=1, how="all")

    columns = list(df.columns)

    machinery_cols = find_matching_columns(columns, MACHINERY_KEYWORDS)
    location_cols = find_matching_columns(columns, LOCATION_KEYWORDS)
    brand_cols = find_matching_columns(columns, BRAND_KEYWORDS)
    model_cols = find_matching_columns(columns, MODEL_KEYWORDS)
    power_cols = find_matching_columns(columns, POWER_KEYWORDS)
    quantity_cols = find_matching_columns(columns, QUANTITY_KEYWORDS)
    owner_cols = find_matching_columns(columns, OWNER_KEYWORDS)
    date_cols = find_matching_columns(columns, DATE_KEYWORDS)

    duplicate_rows = df.duplicated().sum()

    usability_rating = classify_usability(
        row_count=len(df),
        col_count=len(df.columns),
        machinery_cols=machinery_cols,
        location_cols=location_cols
    )

    file_summaries.append({
        "file_name": file_path.name,
        "file_path": str(file_path),
        "inferred_region": inferred_region,
        "read_status": status,
        "row_count": len(df),
        "column_count": len(df.columns),
        "duplicate_rows": duplicate_rows,
        "machinery_columns": ", ".join(map(str, machinery_cols)),
        "location_columns": ", ".join(map(str, location_cols)),
        "brand_columns": ", ".join(map(str, brand_cols)),
        "model_columns": ", ".join(map(str, model_cols)),
        "power_columns": ", ".join(map(str, power_cols)),
        "quantity_columns": ", ".join(map(str, quantity_cols)),
        "owner_columns": ", ".join(map(str, owner_cols)),
        "date_columns": ", ".join(map(str, date_cols)),
        "usability_rating": usability_rating
    })

    # Column summary
    for col in df.columns:
        non_missing = df[col].notna().sum()
        unique_count = df[col].nunique(dropna=True)

        example_values = (
            df[col]
            .dropna()
            .astype(str)
            .head(5)
            .tolist()
        )

        column_summaries.append({
            "file_name": file_path.name,
            "inferred_region": inferred_region,
            "column_name": col,
            "normalized_column_name": normalize_colname(col),
            "non_missing_count": non_missing,
            "missing_count": df[col].isna().sum(),
            "missing_rate": round(df[col].isna().mean(), 4),
            "unique_count": unique_count,
            "example_values": " | ".join(example_values)
        })

    # Sample first 5 rows per file
    sample = df.head(5).copy()
    sample.insert(0, "source_file", file_path.name)
    sample.insert(1, "inferred_region", inferred_region)
    sample_rows.append(sample)

# ============================================================
# COMBINE OUTPUTS
# ============================================================

file_summary_df = pd.DataFrame(file_summaries)
column_summary_df = pd.DataFrame(column_summaries)

if sample_rows:
    sample_rows_df = pd.concat(sample_rows, ignore_index=True, sort=False)
else:
    sample_rows_df = pd.DataFrame()

# Overall summary
overall_summary = pd.DataFrame([{
    "total_files_found": len(all_files),
    "readable_files": (file_summary_df["read_status"] == "OK").sum() if not file_summary_df.empty else 0,
    "total_rows": file_summary_df["row_count"].sum() if "row_count" in file_summary_df else 0,
    "total_columns_scanned": file_summary_df["column_count"].sum() if "column_count" in file_summary_df else 0,
    "good_files": file_summary_df["usability_rating"].astype(str).str.contains("GOOD", na=False).sum() if not file_summary_df.empty else 0,
    "partial_files": file_summary_df["usability_rating"].astype(str).str.contains("PARTIAL", na=False).sum() if not file_summary_df.empty else 0,
    "low_or_unusable_files": (
        file_summary_df["usability_rating"].astype(str).str.contains("LOW|UNUSABLE", na=False).sum()
        if not file_summary_df.empty else 0
    )
}])

# ============================================================
# SAVE EXCEL DIAGNOSTIC OUTPUT
# ============================================================

with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
    overall_summary.to_excel(writer, sheet_name="Overall Summary", index=False)
    file_summary_df.to_excel(writer, sheet_name="File Summary", index=False)
    column_summary_df.to_excel(writer, sheet_name="Column Summary", index=False)
    sample_rows_df.to_excel(writer, sheet_name="Sample Rows", index=False)

print("\nDONE.")
print("Diagnostic file saved to:")
print(OUTPUT_FILE)

print("\nOverall Summary:")
print(overall_summary)

### 3B. ABEMIS Fuel-Relevance Separator

Splits inventory rows into fuel-consuming vs. non-fuel machinery using keyword rules.

**Why a priority-ordered keyword chain?**

ABEMIS mixes diesel tractors with electric mills, irrigation pumps, GMP-certified food processors, and storage units.

* Simple "contains 'tractor'" filter misses combine harvesters
* Simple approach accidentally includes electric tractors
* **Chain enforces precedence:**
  1. Any `NO_FUEL` keyword → non-fuel ("electric tractor" → non-fuel)
  2. Then `FUEL` keywords
  3. Then `UNCERTAIN` for rows with rated power but no keyword match
  4. Ambiguous rows stay `UNCERTAIN` for manual review

In [ ]:
# ============================================================
# ABEMIS INVENTORY FUEL-RELEVANCE SEPARATOR - IMPROVED V2
# ============================================================

# ============================================================
# FILE LOCATIONS
# ============================================================

OUTPUT_FILE = ABEMIS_FUEL_DIR / "ABEMIS_Fuel_vs_NoFuel_Machinery_V2.xlsx"

# ============================================================
# COLUMN SETTINGS
# ============================================================

POSSIBLE_MACHINE_COLUMNS = [
    "Machine Name",
    "Machinery Name",
    "Machine",
    "Machinery",
    "Equipment",
    "Equipment Name",
    "Name of Machine",
]

POSSIBLE_POWER_COLUMNS = [
    "Rated Power",
    "Power",
    "Horsepower",
    "HP",
    "kW",
]

# ============================================================
# KEYWORD RULES
# ============================================================

FUEL_KEYWORDS = [
    "tractor",
    "four wheel tractor",
    "four-wheel tractor",
    "4wt",
    "hand tractor",
    "walking type",
    "walking-type",
    "power tiller",
    "rotary tiller",
    "rotavator",
    "combine harvester",
    "harvester",
    "reaper",
    "transplanter",
    "water pump",
    "irrigation pump",
    "pump",
    "engine",
    "diesel",
    "gasoline",
    "petrol",
    "kerosene",
    "dryer",
    "mechanical dryer",
    "flatbed dryer",
    "recirculating dryer",
    "thresher",
    "sheller",
    "rice mill",
    "milling machine",
    "corn mill",
    "sprayer",
    "shredder",
    "chipper",
    "grass cutter",
    "brush cutter",
    "lawn mower",
    "hauling truck",
    "truck",
    "cargo motorcycle",
    "three wheel cargo motorcycle",
    "motorcycle",
    "multi cultivator",
    "cultivator",
    "disc cultivator",
    "inter row tine cultivator",
    "chainsaw",
    "mist blower",
    "pole pruner",
    "backhoe loader",
    "front loader",
    "loader",
    "forklift",
    "heavy duty forklift",
    "rough terrain crane",
    "telehandler",
    "drilling rig",
    "soil auger",
    "trailer",
    "hauler",
    "paddy hauler",
    "granule applicator",
    "fertilizer applicator",
    "broadcast spreader",
    "seed spreader",
    "corn seeder",
    "mechanical corn seeder",
    "precision seeder",
    "precision planter",
    "riding type palay seeder",
    "pneumatic corn seeder",
    "drum seeder",
    "sugarcane planter",
    "rotary ditcher",
    "mouldboard plow",
    "cane gruber",
    "cassava rootcrop digger",
    "power duster",
    "fogging machine",
    "refrigerated van",
    "mobile veterinary clinic",
    "agriculture promotion vehicle",
    "chipping machine",
    "weeder",
    "generator",
    "seeder",
    "corn picker",
    "baler",
    "manure rotary spreader",
    "compost windrow turner",
]

NO_FUEL_KEYWORDS = [
    # electric-powered / grid-powered
    "electric",
    "electrical",
    "electric motor",
    "motor driven",
    "motor-driven",
    "single phase",
    "three phase",
    "1 phase",
    "3 phase",
    "220v",
    "230v",
    "240v",
    "380v",
    "440v",

    # GMP and food-processing facility indicators
    "gmp",
    "good manufacturing practices",
    "food grade",
    "food-grade",
    "food processing",
    "processing facility",
    "processing center",
    "processing plant",
    "postharvest facility",
    "stainless",
    "stainless steel",
    "ss table",
    "working table",
    "preparation table",
    "washing table",

    # packaging / bottling / sealing
    "packaging",
    "packing",
    "vacuum sealer",
    "sealer",
    "impulse sealer",
    "continuous band sealer",
    "bottling",
    "bottle",
    "capping",
    "capper",
    "labeling",
    "labeler",
    "filling machine",
    "filler",
    "strapping",
    "strapping machine",

    # food processing usually electric/GMP-related
    "mixing machine",
    "mixer",
    "pulverizer",
    "grinder",
    "slicer",
    "chopper",
    "washer",
    "blancher",
    "dehydrator",
    "freezer",
    "chiller",
    "cold storage",
    "cabinet dryer",
    "dryer cabinet",
    "oven",
    "roaster",
    "steamer",
    "pasteurizer",
    "kettle",
    "juicer",
    "extractor",
    "separator",
    "peeler",
    "crusher",

    # infrastructure / facility / non-machinery
    "hydroponics",
    "greenhouse",
    "nursery",
    "building",
    "warehouse",
    "storage",
    "solar",
    "solar-powered",
    "sensor",
    "moisture meter",
    "weighing scale",
    "scale",
    "tester",
    "analyzer",
    "laboratory",
    "meter",
    "drone",
    "uav",
    "software",
    "computer",
    "printer",
    "tablet",
    "gps",
    "office",
    "facility",
    "structure",
    "shed",
    "trays",
    "net",
    "plastic",
    "irrigation system",
    "drip irrigation",
    "sprinkler system",
    "photovoltaic system",
    "egg incubator",
    "rotary composter",
    "rotary sifter",
    "vermicast sifter",
    "vermi tea brewer",
    "misting system",
    "automated disinfection system",
    "ln2 tank",
    "hermetic bag",
    "ice making machine",
    "pressure cooker",
    "abaca stripper",
    "fiber decorticator",
    "coconut coir decorticator",
    "hammer mill",
    "micro mill",
    "feed mill",
    "flour mill",
    "adlay mill",
    "coffee pulper",
    "coffee huller",
    "cacao huller",
    "paddy huller",
    "brown rice huller",
    "impeller type huller",
    "rubber roll type huller",
    "huller",
    "mist polisher",
    "rice polisher",
    "polisher",
    "rice whitener",
    "whitener",
    "destoner",
    "color sorter",
    "soybean sorter",
    "seed sorter",
    "length grader",
    "paddy seed cleaner",
    "pre cleaner",
    "vibrating cleaner",
    "corn grain cleaner",
    "grain collector",
    "cacao cracker and winnower",
    "cassava granulator",
    "soybean granulator",
    "cassava grater",
    "multi commodity grater",
    "multi commodity grinding machine",
    "poultry defeathering machine",
    "milking machine",
    "milk homogenizer",
    "sack sewing machine",
    "cotton ginning machine",
    "roasting machine",
    "melanger",
    "chocolate tempering machine",
    "mushroom bagger",
    "feed pelletizer",
    "fertilizer pelletizer",
    "feed pellet cooler",
    "briquetting machine",
    "tramline system",
    "vari trac wheels",
    "vari-trac wheels",
    "paddy vari trac wheels",
    "aerator",
]

UNCERTAIN_KEYWORDS = [
    "equipment",
    "system",
    "set",
    "package",
    "accessories",
    "machinery",
]

# ============================================================
# HELPER FUNCTIONS
# ============================================================

def normalize_text(x):
    if pd.isna(x):
        return ""

    x = str(x).lower().strip()
    x = re.sub(r"[^a-z0-9]+", " ", x)
    x = re.sub(r"\s+", " ", x)
    return x.strip()

def keyword_in_text(keyword, text):
    key = normalize_text(keyword)
    return key in text

def find_column(df, possible_names):
    normalized_columns = {
        normalize_text(col): col for col in df.columns
    }

    for name in possible_names:
        name_norm = normalize_text(name)

        if name_norm in normalized_columns:
            return normalized_columns[name_norm]

    for col in df.columns:
        col_norm = normalize_text(col)

        for name in possible_names:
            name_norm = normalize_text(name)

            if name_norm in col_norm:
                return col

    return None

def infer_region_from_filename(file_path):
    name = file_path.stem.upper()

    match = re.search(r"(NCR|CAR|BARMM|NIRR|R\d+[A-Z]?)", name)

    if match:
        return match.group(1)

    return None

def has_valid_power(power_value):
    if power_value is None or pd.isna(power_value):
        return False

    try:
        power_num = float(str(power_value).replace(",", "").strip())
        return power_num > 0
    except Exception:
        return False

def classify_fuel_relevance(machine_name, power_value=None):
    text = normalize_text(machine_name)

    matched_fuel = [kw for kw in FUEL_KEYWORDS if keyword_in_text(kw, text)]
    matched_no_fuel = [kw for kw in NO_FUEL_KEYWORDS if keyword_in_text(kw, text)]
    matched_uncertain = [kw for kw in UNCERTAIN_KEYWORDS if keyword_in_text(kw, text)]

    has_power = has_valid_power(power_value)

    # Rule 1: electric/GMP/food processing/no-fuel terms override most matches.
    # Example: "Electric Grinder" should be NO_FUEL, not fuel relevant.
    if matched_no_fuel:
        return (
            "NO_FUEL_OR_NON_RELEVANT",
            ", ".join(matched_fuel),
            ", ".join(matched_no_fuel),
            "NO_FUEL_OVERRIDE"
        )

    # Rule 2: clear fuel-relevant machinery
    if matched_fuel:
        return (
            "FUEL_RELEVANT",
            ", ".join(matched_fuel),
            "",
            "FUEL_KEYWORD_MATCH"
        )

    # Rule 3: has power but no clear label
    if has_power:
        return (
            "UNCERTAIN_REVIEW",
            "has rated power but unknown machinery",
            "",
            "HAS_POWER_BUT_UNKNOWN"
        )

    # Rule 4: ambiguous terms
    if matched_uncertain:
        return (
            "UNCERTAIN_REVIEW",
            "",
            ", ".join(matched_uncertain),
            "AMBIGUOUS_KEYWORD"
        )

    return (
        "NO_FUEL_OR_NON_RELEVANT",
        "",
        "",
        "NO_MATCH_DEFAULT_NO_FUEL"
    )

def read_file_safely(file_path):
    suffix = file_path.suffix.lower()

    try:
        if suffix in [".xlsx", ".xls"]:
            return pd.read_excel(file_path), "OK"

        if suffix == ".csv":
            try:
                return pd.read_csv(file_path, encoding="utf-8"), "OK"
            except UnicodeDecodeError:
                return pd.read_csv(file_path, encoding="latin1"), "OK"

        return None, f"Unsupported file type: {suffix}"

    except Exception as e:
        return None, f"READ_ERROR: {e}"

# ============================================================
# MAIN PROCESS
# ============================================================

all_files = []

for ext in ["*.xlsx", "*.xls", "*.csv"]:
    all_files.extend(ABEMIS_EXTRACTED_DIR.rglob(ext))

all_files = sorted(all_files)

print("Files found:", len(all_files))

records = []
file_logs = []

for i, file_path in enumerate(all_files, start=1):
    print(f"[{i}/{len(all_files)}] Processing {file_path.name}")

    df = None
    df, status = read_file_safely(file_path)

    if df is None:
        file_logs.append({
            "file_name": file_path.name,
            "status": status,
            "rows": 0,
            "machine_column": None,
            "power_column": None
        })
        continue

    df = df.dropna(how="all").dropna(axis=1, how="all")

    machine_col = find_column(df, POSSIBLE_MACHINE_COLUMNS)
    power_col = find_column(df, POSSIBLE_POWER_COLUMNS)

    region = infer_region_from_filename(file_path)

    if machine_col is None:
        file_logs.append({
            "file_name": file_path.name,
            "status": "NO_MACHINE_COLUMN_FOUND",
            "rows": len(df),
            "machine_column": None,
            "power_column": power_col
        })
        continue

    for _, row in df.iterrows():
        machine_name = row.get(machine_col, None)
        power_value = row.get(power_col, None) if power_col else None

        classification, matched_fuel, matched_no_fuel, rule_applied = classify_fuel_relevance(
            machine_name,
            power_value
        )

        record = row.to_dict()
        record["source_file"] = file_path.name
        record["inferred_region"] = region
        record["detected_machine_column"] = machine_col
        record["detected_power_column"] = power_col
        record["machine_name_for_classification"] = machine_name
        record["fuel_relevance_class"] = classification
        record["matched_fuel_keywords"] = matched_fuel
        record["matched_no_fuel_keywords"] = matched_no_fuel
        record["classification_rule_applied"] = rule_applied

        records.append(record)

    file_logs.append({
        "file_name": file_path.name,
        "status": "OK",
        "rows": len(df),
        "machine_column": machine_col,
        "power_column": power_col
    })

# ============================================================
# OUTPUT DATAFRAMES
# ============================================================

master = pd.DataFrame(records)
file_log = pd.DataFrame(file_logs)

fuel_relevant = master[
    master["fuel_relevance_class"] == "FUEL_RELEVANT"
].copy()

no_fuel = master[
    master["fuel_relevance_class"] == "NO_FUEL_OR_NON_RELEVANT"
].copy()

uncertain = master[
    master["fuel_relevance_class"] == "UNCERTAIN_REVIEW"
].copy()

summary = pd.DataFrame([{
    "total_files_processed": len(all_files),
    "total_records": len(master),
    "fuel_relevant_records": len(fuel_relevant),
    "no_fuel_or_non_relevant_records": len(no_fuel),
    "uncertain_review_records": len(uncertain),
}])

machinery_summary = (
    master
    .groupby(
        [
            "machine_name_for_classification",
            "fuel_relevance_class",
            "classification_rule_applied"
        ],
        dropna=False
    )
    .size()
    .reset_index(name="records")
    .sort_values("records", ascending=False)
)

rule_summary = (
    master
    .groupby(["fuel_relevance_class", "classification_rule_applied"], dropna=False)
    .size()
    .reset_index(name="records")
    .sort_values("records", ascending=False)
)

# ============================================================
# SAVE OUTPUT
# ============================================================

with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
    summary.to_excel(writer, sheet_name="Summary", index=False)
    rule_summary.to_excel(writer, sheet_name="Rule Summary", index=False)
    file_log.to_excel(writer, sheet_name="File Log", index=False)
    machinery_summary.to_excel(writer, sheet_name="Machinery Summary", index=False)
    fuel_relevant.to_excel(writer, sheet_name="Fuel Relevant", index=False)
    no_fuel.to_excel(writer, sheet_name="No Fuel", index=False)
    uncertain.to_excel(writer, sheet_name="Uncertain Review", index=False)
    master.to_excel(writer, sheet_name="Master Classified", index=False)

print("\nDONE.")
print("Output saved to:")
print(OUTPUT_FILE)

print("\nSummary:")
print(summary)

### 3C. AMTEC Analytics & Cleaning

Cleans extracted AMTEC data, computes derived metrics (fuel intensity, power classes), flags outliers, and produces summary tables and charts.

**Data normalization:**

* Fuel in liters/hour and rated power in kW
* Compute intensity: 
$$I_f = \frac{\text{L/h}}{\text{kW}}$$
* Enables comparison across power classes

**Outlier detection:**

* Studentized residuals against preliminary OLS
* FINAL modeling excludes pathological records (mis-transcribed decimals, wrong units)
* Prevents outliers from dominating the fit

In [ ]:
INPUT_FILE = AMTEC_EXTRACTION_DIR / "AMTEC_full_extracted_dataset.xlsx"
OUTPUT_DIR = AMTEC_ANALYTICS_DIR
OUTPUT_EXCEL = AMTEC_ANALYTICS_DIR / "AMTEC_Test_Report_Fuel_Power_Analytics_V2.xlsx"

# ============================================================
# AMTEC TEST REPORT ANALYTICS SCRIPT - IMPROVED V2
# Fuel, Power, Machinery Family, Regression, Outlier Analytics
# ============================================================

# ============================================================
# FILE LOCATIONS
# ============================================================

# ============================================================
# LOAD DATA
# ============================================================

df = pd.read_excel(INPUT_FILE)

print("Original records:", len(df))

# ============================================================
# BASIC CLEANING
# ============================================================

text_cols = [
    "machinery_type",
    "fuel_unit",
    "brand",
    "model",
    "fuel_type",
    "project_relevance",
    "extraction_status"
]

for col in text_cols:
    if col in df.columns:
        df[col] = df[col].astype("string").str.strip()

numeric_cols = [
    "power_kw",
    "power_hp",
    "fuel_value",
    "field_capacity_value",
    "operating_speed_value",
    "general_capacity_value"
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# ============================================================
# FILTER COMPARABLE FUEL DATA
# ============================================================

usable = df[
    df["machinery_type"].notna() &
    df["power_kw"].notna() &
    df["fuel_value"].notna()
].copy()

usable["fuel_unit_clean"] = (
    usable["fuel_unit"]
    .astype("string")
    .str.lower()
    .str.replace("l/hr", "l/h", regex=False)
    .str.replace("li/hr", "l/h", regex=False)
    .str.replace("liter/hr", "l/h", regex=False)
    .str.strip()
)

usable = usable[usable["fuel_unit_clean"].isin(["l/h"])].copy()

usable["fuel_l_per_hr"] = usable["fuel_value"]

# Remove physically impossible values
usable = usable[
    (usable["power_kw"] > 0) &
    (usable["power_kw"] <= 500) &
    (usable["fuel_l_per_hr"] > 0) &
    (usable["fuel_l_per_hr"] <= 300)
].copy()

print("Usable L/h records:", len(usable))

# ============================================================
# MACHINERY FAMILY CLASSIFICATION
# ============================================================

def classify_machinery_family(machine):
    if pd.isna(machine):
        return "Unknown"

    machine = str(machine).strip()

    mobile_field = [
        "Four-Wheel Tractor",
        "Hand Tractor",
        "Walking-Type Agricultural Tractor",
        "Rotary Tiller",
        "Rice Transplanter",
        "Reaper",
        "Seeder",
        "Sprayer",
    ]

    harvest = [
        "Combine Harvester",
    ]

    stationary_engine = [
        "Small Engine",
        "Water Pump",
        "Solar-Powered Irrigation System",
    ]

    postharvest_processing = [
        "Thresher",
        "Sheller",
        "Rice Mill",
        "Mechanical Dryer",
    ]

    if machine in mobile_field:
        return "Mobile Field Machinery"

    if machine in harvest:
        return "Harvest Machinery"

    if machine in stationary_engine:
        return "Stationary Engine / Irrigation"

    if machine in postharvest_processing:
        return "Postharvest / Processing"

    return "Other / Unclassified"

def classify_analysis_subset(machine):
    family = classify_machinery_family(machine)

    if family in ["Mobile Field Machinery", "Harvest Machinery"]:
        return "Dataset A - Field Machinery"

    if family in ["Stationary Engine / Irrigation", "Postharvest / Processing"]:
        return "Dataset B - Stationary and Processing"

    return "Dataset C - Other"

usable["machinery_family"] = usable["machinery_type"].apply(classify_machinery_family)
usable["analysis_subset"] = usable["machinery_type"].apply(classify_analysis_subset)

# ============================================================
# DERIVED ENGINEERING METRICS
# ============================================================

usable["fuel_l_per_kw_hr"] = usable["fuel_l_per_hr"] / usable["power_kw"]
usable["kw_per_l_per_hr"] = usable["power_kw"] / usable["fuel_l_per_hr"]

usable["fuel_l_per_hp_hr"] = usable["fuel_l_per_hr"] / usable["power_hp"]
usable["hp_per_l_per_hr"] = usable["power_hp"] / usable["fuel_l_per_hr"]

# ============================================================
# POWER CLASSIFICATION
# ============================================================

def fixed_power_class_kw(power_kw):
    if pd.isna(power_kw):
        return np.nan
    elif power_kw < 5:
        return "Very Low Power (<5 kW)"
    elif power_kw < 10:
        return "Low Power (5–<10 kW)"
    elif power_kw < 20:
        return "Medium Power (10–<20 kW)"
    elif power_kw < 40:
        return "High Power (20–<40 kW)"
    elif power_kw < 75:
        return "Very High Power (40–<75 kW)"
    else:
        return "Extra High Power (≥75 kW)"

usable["power_class_fixed"] = usable["power_kw"].apply(fixed_power_class_kw)

def add_within_machine_power_class(group):
    group = group.copy()

    if group["power_kw"].nunique() < 3 or len(group) < 6:
        group["power_class_within_machine"] = "Single/Narrow Power Range"
        return group

    try:
        group["power_class_within_machine"] = pd.qcut(
            group["power_kw"],
            q=3,
            labels=[
                "Low Power Within Type",
                "Medium Power Within Type",
                "High Power Within Type",
            ],
            duplicates="drop"
        ).astype(str)
    except Exception:
        group["power_class_within_machine"] = "Single/Narrow Power Range"

    return group

usable = (
    usable
    .groupby("machinery_type", group_keys=False)
    .apply(add_within_machine_power_class)
)

# ============================================================
# FUEL EFFICIENCY CLASSIFICATION
# ============================================================

def fuel_intensity_class(value):
    """
    Uses fuel_l_per_kw_hr.
    Lower means less fuel per unit rated power.
    """
    if pd.isna(value):
        return np.nan
    elif value < 0.10:
        return "Very Low Fuel Intensity (<0.10 L/kW-h)"
    elif value < 0.20:
        return "Low Fuel Intensity (0.10–<0.20 L/kW-h)"
    elif value < 0.35:
        return "Moderate Fuel Intensity (0.20–<0.35 L/kW-h)"
    elif value < 0.60:
        return "High Fuel Intensity (0.35–<0.60 L/kW-h)"
    else:
        return "Very High Fuel Intensity (≥0.60 L/kW-h)"

usable["fuel_intensity_class"] = usable["fuel_l_per_kw_hr"].apply(fuel_intensity_class)

# ============================================================
# OUTLIER SEVERITY
# ============================================================

def add_outlier_severity(group):
    group = group.copy()

    if len(group) < 5:
        group["fuel_z_score_within_machine"] = np.nan
        group["fuel_outlier_severity"] = "Insufficient Records"
        return group

    mean_val = group["fuel_l_per_hr"].mean()
    std_val = group["fuel_l_per_hr"].std()

    if std_val == 0 or pd.isna(std_val):
        group["fuel_z_score_within_machine"] = 0
        group["fuel_outlier_severity"] = "No Variation"
        return group

    group["fuel_z_score_within_machine"] = (
        (group["fuel_l_per_hr"] - mean_val) / std_val
    )

    abs_z = group["fuel_z_score_within_machine"].abs()

    group["fuel_outlier_severity"] = np.select(
        [
            abs_z >= 3,
            abs_z >= 2,
            abs_z >= 1.5,
        ],
        [
            "Extreme",
            "Moderate",
            "Mild",
        ],
        default="Normal"
    )

    return group

usable = (
    usable
    .groupby("machinery_type", group_keys=False)
    .apply(add_outlier_severity)
)

possible_outliers = usable[
    usable["fuel_outlier_severity"].isin(["Mild", "Moderate", "Extreme"])
].copy()

# ============================================================
# REGRESSION FUNCTIONS
# ============================================================

def regression_summary(group, group_name_col):
    results = []

    for name, g in group:
        g = g.dropna(subset=["power_kw", "fuel_l_per_hr"]).copy()

        if len(g) < 5 or g["power_kw"].nunique() < 2:
            results.append({
                group_name_col: name,
                "records": len(g),
                "regression_status": "Insufficient data",
                "intercept": np.nan,
                "slope_fuel_per_kw": np.nan,
                "equation": None,
                "r2": np.nan,
                "mae": np.nan,
                "rmse": np.nan,
            })
            continue

        X = g[["power_kw"]].values
        y = g["fuel_l_per_hr"].values

        model = LinearRegression()
        model.fit(X, y)

        pred = model.predict(X)

        intercept = float(model.intercept_)
        slope = float(model.coef_[0])

        mse = mean_squared_error(y, pred)

        results.append({
            group_name_col: name,
            "records": len(g),
            "regression_status": "OK",
            "intercept": intercept,
            "slope_fuel_per_kw": slope,
            "equation": f"Fuel_L_h = {intercept:.4f} + {slope:.4f}(Power_kW)",
            "r2": r2_score(y, pred),
            "mae": mean_absolute_error(y, pred),
            "rmse": np.sqrt(mse),
        })

    return pd.DataFrame(results)

regression_by_machine = regression_summary(
    usable.groupby("machinery_type"),
    "machinery_type"
).sort_values(["regression_status", "records"], ascending=[True, False])

regression_by_family = regression_summary(
    usable.groupby("machinery_family"),
    "machinery_family"
).sort_values(["regression_status", "records"], ascending=[True, False])

regression_by_subset = regression_summary(
    usable.groupby("analysis_subset"),
    "analysis_subset"
).sort_values(["regression_status", "records"], ascending=[True, False])

# ============================================================
# CORRELATION TABLES
# ============================================================

corr_cols = [
    "power_kw",
    "power_hp",
    "fuel_l_per_hr",
    "fuel_l_per_kw_hr",
    "field_capacity_value",
    "operating_speed_value",
    "general_capacity_value",
]

corr_cols = [c for c in corr_cols if c in usable.columns]

overall_correlation = usable[corr_cols].corr(numeric_only=True).reset_index()
overall_correlation = overall_correlation.rename(columns={"index": "variable"})

fuel_correlation = (
    usable[corr_cols]
    .corr(numeric_only=True)[["fuel_l_per_hr"]]
    .reset_index()
    .rename(columns={"index": "variable", "fuel_l_per_hr": "correlation_with_fuel_l_hr"})
    .sort_values("correlation_with_fuel_l_hr", ascending=False)
)

def correlation_by_group(data, group_col):
    rows = []

    for name, g in data.groupby(group_col):
        g = g[corr_cols].dropna(how="all")

        if len(g) < 5:
            rows.append({
                group_col: name,
                "records": len(g),
                "corr_power_kw_fuel": np.nan,
                "corr_field_capacity_fuel": np.nan,
                "corr_speed_fuel": np.nan,
            })
            continue

        corr = g.corr(numeric_only=True)

        rows.append({
            group_col: name,
            "records": len(g),
            "corr_power_kw_fuel": corr.loc["power_kw", "fuel_l_per_hr"] if "power_kw" in corr.index and "fuel_l_per_hr" in corr.columns else np.nan,
            "corr_field_capacity_fuel": corr.loc["field_capacity_value", "fuel_l_per_hr"] if "field_capacity_value" in corr.index and "fuel_l_per_hr" in corr.columns else np.nan,
            "corr_speed_fuel": corr.loc["operating_speed_value", "fuel_l_per_hr"] if "operating_speed_value" in corr.index and "fuel_l_per_hr" in corr.columns else np.nan,
        })

    return pd.DataFrame(rows)

correlation_by_machine = correlation_by_group(usable, "machinery_type")
correlation_by_family = correlation_by_group(usable, "machinery_family")

# ============================================================
# SUMMARY TABLES
# ============================================================

record_cols = [
    "test_report_no",
    "year",
    "machinery_family",
    "analysis_subset",
    "machinery_type",
    "brand",
    "model",
    "rated_power_raw",
    "power_kw",
    "power_hp",
    "power_class_fixed",
    "power_class_within_machine",
    "fuel_type",
    "fuel_consumption_raw",
    "fuel_l_per_hr",
    "fuel_l_per_kw_hr",
    "fuel_l_per_hp_hr",
    "kw_per_l_per_hr",
    "fuel_intensity_class",
    "field_capacity_value",
    "field_capacity_unit",
    "operating_speed_value",
    "operating_speed_unit",
    "general_capacity_value",
    "general_capacity_unit",
    "fuel_z_score_within_machine",
    "fuel_outlier_severity",
    "source_file",
    "source_path"
]

record_cols = [c for c in record_cols if c in usable.columns]
clean_record_level = usable[record_cols].copy()

summary_by_machine = (
    usable
    .groupby(["machinery_family", "machinery_type"])
    .agg(
        records=("fuel_l_per_hr", "count"),
        avg_power_kw=("power_kw", "mean"),
        min_power_kw=("power_kw", "min"),
        max_power_kw=("power_kw", "max"),
        avg_power_hp=("power_hp", "mean"),
        avg_fuel_l_hr=("fuel_l_per_hr", "mean"),
        min_fuel_l_hr=("fuel_l_per_hr", "min"),
        max_fuel_l_hr=("fuel_l_per_hr", "max"),
        median_fuel_l_hr=("fuel_l_per_hr", "median"),
        avg_fuel_l_per_kw_hr=("fuel_l_per_kw_hr", "mean"),
        median_fuel_l_per_kw_hr=("fuel_l_per_kw_hr", "median"),
        avg_kw_per_l_per_hr=("kw_per_l_per_hr", "mean")
    )
    .reset_index()
    .sort_values(["machinery_family", "records"], ascending=[True, False])
)

summary_by_family = (
    usable
    .groupby("machinery_family")
    .agg(
        records=("fuel_l_per_hr", "count"),
        machinery_types=("machinery_type", "nunique"),
        avg_power_kw=("power_kw", "mean"),
        min_power_kw=("power_kw", "min"),
        max_power_kw=("power_kw", "max"),
        avg_fuel_l_hr=("fuel_l_per_hr", "mean"),
        min_fuel_l_hr=("fuel_l_per_hr", "min"),
        max_fuel_l_hr=("fuel_l_per_hr", "max"),
        median_fuel_l_hr=("fuel_l_per_hr", "median"),
        avg_fuel_l_per_kw_hr=("fuel_l_per_kw_hr", "mean"),
    )
    .reset_index()
    .sort_values("records", ascending=False)
)

summary_by_subset = (
    usable
    .groupby("analysis_subset")
    .agg(
        records=("fuel_l_per_hr", "count"),
        machinery_types=("machinery_type", "nunique"),
        avg_power_kw=("power_kw", "mean"),
        avg_fuel_l_hr=("fuel_l_per_hr", "mean"),
        avg_fuel_l_per_kw_hr=("fuel_l_per_kw_hr", "mean"),
    )
    .reset_index()
    .sort_values("records", ascending=False)
)

summary_by_fixed_power_class = (
    usable
    .groupby(["machinery_family", "machinery_type", "power_class_fixed"])
    .agg(
        records=("fuel_l_per_hr", "count"),
        avg_power_kw=("power_kw", "mean"),
        min_power_kw=("power_kw", "min"),
        max_power_kw=("power_kw", "max"),
        avg_fuel_l_hr=("fuel_l_per_hr", "mean"),
        min_fuel_l_hr=("fuel_l_per_hr", "min"),
        max_fuel_l_hr=("fuel_l_per_hr", "max"),
        median_fuel_l_hr=("fuel_l_per_hr", "median"),
        avg_fuel_l_per_kw_hr=("fuel_l_per_kw_hr", "mean"),
    )
    .reset_index()
    .sort_values(["machinery_family", "machinery_type", "avg_power_kw"])
)

summary_by_within_power_class = (
    usable
    .groupby(["machinery_family", "machinery_type", "power_class_within_machine"])
    .agg(
        records=("fuel_l_per_hr", "count"),
        avg_power_kw=("power_kw", "mean"),
        min_power_kw=("power_kw", "min"),
        max_power_kw=("power_kw", "max"),
        avg_fuel_l_hr=("fuel_l_per_hr", "mean"),
        min_fuel_l_hr=("fuel_l_per_hr", "min"),
        max_fuel_l_hr=("fuel_l_per_hr", "max"),
        median_fuel_l_hr=("fuel_l_per_hr", "median"),
        avg_fuel_l_per_kw_hr=("fuel_l_per_kw_hr", "mean"),
    )
    .reset_index()
    .sort_values(["machinery_family", "machinery_type", "avg_power_kw"])
)

fuel_intensity_summary = (
    usable
    .groupby(["machinery_family", "machinery_type", "fuel_intensity_class"])
    .agg(
        records=("fuel_l_per_hr", "count"),
        avg_power_kw=("power_kw", "mean"),
        avg_fuel_l_hr=("fuel_l_per_hr", "mean"),
        avg_fuel_l_per_kw_hr=("fuel_l_per_kw_hr", "mean"),
    )
    .reset_index()
    .sort_values(["machinery_family", "machinery_type", "avg_fuel_l_per_kw_hr"])
)

pivot_avg_fuel_fixed_power = pd.pivot_table(
    usable,
    values="fuel_l_per_hr",
    index=["machinery_family", "machinery_type"],
    columns="power_class_fixed",
    aggfunc="mean"
).reset_index()

pivot_count_fixed_power = pd.pivot_table(
    usable,
    values="fuel_l_per_hr",
    index=["machinery_family", "machinery_type"],
    columns="power_class_fixed",
    aggfunc="count"
).reset_index()

# ============================================================
# DATASET SPLITS
# ============================================================

dataset_a_field = usable[
    usable["analysis_subset"] == "Dataset A - Field Machinery"
].copy()

dataset_b_stationary = usable[
    usable["analysis_subset"] == "Dataset B - Stationary and Processing"
].copy()

weak_categories = (
    usable
    .groupby("machinery_type")
    .size()
    .reset_index(name="records")
)

weak_categories = weak_categories[weak_categories["records"] < 5]

# ============================================================
# SAVE EXCEL OUTPUT
# ============================================================

with pd.ExcelWriter(OUTPUT_EXCEL, engine="openpyxl") as writer:
    clean_record_level.to_excel(writer, sheet_name="Clean Record Level", index=False)
    summary_by_family.to_excel(writer, sheet_name="Summary by Family", index=False)
    summary_by_subset.to_excel(writer, sheet_name="Summary by Dataset", index=False)
    summary_by_machine.to_excel(writer, sheet_name="Summary by Machine", index=False)
    summary_by_fixed_power_class.to_excel(writer, sheet_name="By Fixed Power Class", index=False)
    summary_by_within_power_class.to_excel(writer, sheet_name="By Within-Type Power", index=False)
    fuel_intensity_summary.to_excel(writer, sheet_name="Fuel Intensity Summary", index=False)
    regression_by_machine.to_excel(writer, sheet_name="Regression by Machine", index=False)
    regression_by_family.to_excel(writer, sheet_name="Regression by Family", index=False)
    regression_by_subset.to_excel(writer, sheet_name="Regression by Dataset", index=False)
    fuel_correlation.to_excel(writer, sheet_name="Fuel Correlation", index=False)
    overall_correlation.to_excel(writer, sheet_name="Overall Correlation", index=False)
    correlation_by_machine.to_excel(writer, sheet_name="Correlation by Machine", index=False)
    correlation_by_family.to_excel(writer, sheet_name="Correlation by Family", index=False)
    possible_outliers.to_excel(writer, sheet_name="Outlier Severity", index=False)
    pivot_avg_fuel_fixed_power.to_excel(writer, sheet_name="Pivot Avg Fuel", index=False)
    pivot_count_fixed_power.to_excel(writer, sheet_name="Pivot Count", index=False)
    dataset_a_field.to_excel(writer, sheet_name="Dataset A Field Machinery", index=False)
    dataset_b_stationary.to_excel(writer, sheet_name="Dataset B Stationary", index=False)
    weak_categories.to_excel(writer, sheet_name="Weak Categories", index=False)

print("Excel analytics saved to:")
print(OUTPUT_EXCEL)

# ============================================================
# CHARTS
# ============================================================

# Chart 1: Average fuel by machinery type
chart_data = summary_by_machine.sort_values("avg_fuel_l_hr", ascending=False)

plt.figure(figsize=(12, 6))
plt.bar(chart_data["machinery_type"], chart_data["avg_fuel_l_hr"])
plt.xticks(rotation=75, ha="right")
plt.ylabel("Average Fuel Consumption (L/h)")
plt.xlabel("Machinery Type")
plt.title("Average Fuel Consumption by Machinery Type")
plt.tight_layout()
chart1_path = AMTEC_ANALYTICS_DIR / "average_fuel_by_machinery_type.png"
plt.savefig(chart1_path, dpi=300)
plt.close()

# Chart 2: Average fuel by machinery family
chart_data = summary_by_family.sort_values("avg_fuel_l_hr", ascending=False)

plt.figure(figsize=(10, 6))
plt.bar(chart_data["machinery_family"], chart_data["avg_fuel_l_hr"])
plt.xticks(rotation=45, ha="right")
plt.ylabel("Average Fuel Consumption (L/h)")
plt.xlabel("Machinery Family")
plt.title("Average Fuel Consumption by Machinery Family")
plt.tight_layout()
chart2_path = AMTEC_ANALYTICS_DIR / "average_fuel_by_machinery_family.png"
plt.savefig(chart2_path, dpi=300)
plt.close()

# Chart 3: Rated power vs fuel, all records
plt.figure(figsize=(8, 6))
plt.scatter(usable["power_kw"], usable["fuel_l_per_hr"], alpha=0.7)
plt.xlabel("Rated Power (kW)")
plt.ylabel("Fuel Consumption (L/h)")
plt.title("Rated Power vs Fuel Consumption - All Machinery")
plt.tight_layout()
chart3_path = AMTEC_ANALYTICS_DIR / "power_vs_fuel_all.png"
plt.savefig(chart3_path, dpi=300)
plt.close()

# Chart 4: Rated power vs fuel, Dataset A field machinery
if len(dataset_a_field) > 0:
    plt.figure(figsize=(8, 6))
    plt.scatter(dataset_a_field["power_kw"], dataset_a_field["fuel_l_per_hr"], alpha=0.7)
    plt.xlabel("Rated Power (kW)")
    plt.ylabel("Fuel Consumption (L/h)")
    plt.title("Rated Power vs Fuel Consumption - Field Machinery")
    plt.tight_layout()
    chart4_path = AMTEC_ANALYTICS_DIR / "power_vs_fuel_field_machinery.png"
    plt.savefig(chart4_path, dpi=300)
    plt.close()
else:
    chart4_path = None

# Chart 5: Fuel intensity by machinery type
chart_data = summary_by_machine.sort_values("avg_fuel_l_per_kw_hr", ascending=False)

plt.figure(figsize=(12, 6))
plt.bar(chart_data["machinery_type"], chart_data["avg_fuel_l_per_kw_hr"])
plt.xticks(rotation=75, ha="right")
plt.ylabel("Average Fuel Intensity (L/kW-h)")
plt.xlabel("Machinery Type")
plt.title("Average Fuel Intensity by Machinery Type")
plt.tight_layout()
chart5_path = AMTEC_ANALYTICS_DIR / "fuel_intensity_by_machinery_type.png"
plt.savefig(chart5_path, dpi=300)
plt.close()

print("Charts saved:")
print(chart1_path)
print(chart2_path)
print(chart3_path)
if chart4_path:
    print(chart4_path)
print(chart5_path)

# ============================================================
# SUMMARY TEXT FILE
# ============================================================

summary_file = AMTEC_ANALYTICS_DIR / "AMTEC_Analytics_V2_Summary.txt"

summary_text = f"""
AMTEC TEST REPORT ANALYTICS V2 SUMMARY

Original records: {len(df)}
Usable L/h records: {len(usable)}

Dataset A - Field Machinery records: {len(dataset_a_field)}
Dataset B - Stationary and Processing records: {len(dataset_b_stationary)}

Distinct machinery types: {usable['machinery_type'].nunique()}
Distinct machinery families: {usable['machinery_family'].nunique()}

Possible outliers flagged: {len(possible_outliers)}
Weak machinery categories with less than 5 records: {len(weak_categories)}

Output Excel:
{OUTPUT_EXCEL}

Charts:
{chart1_path}
{chart2_path}
{chart3_path}
{chart4_path}
{chart5_path}
"""

with open(summary_file, "w", encoding="utf-8") as f:
    f.write(summary_text)

print(summary_text)
print("DONE.")

### 3D. AMTEC Regression Modeling

**Regression V3**: Hierarchical OLS (machinery-type → family → global), filters models below minimum R² threshold.

**Why hierarchical?**

* Single global slope underfits high-volume types (tractors, combines) with their own power-fuel curves
* Per-type models overfit on low-count classes
* **Solution:** Assign each machinery_type the most-local model meeting R² threshold (default 0.50, adjusted)
* Otherwise fall back to FAMILY or GLOBAL

In [ ]:
# ============================================================
# AMTEC FUEL REGRESSION PARAMETERS + DIAGNOSTICS V3
# Removes Weak R-squared Models from Final Prediction Hierarchy
# ============================================================

# ============================================================
# INPUT / OUTPUT SETTINGS
# ============================================================

INPUT_FILE = AMTEC_ANALYTICS_DIR / "AMTEC_Test_Report_Fuel_Power_Analytics_V2.xlsx"
SHEET_NAME = "Clean Record Level"
OUTPUT_DIR = AMTEC_REGRESSION_DIR
OUTPUT_EXCEL = AMTEC_REGRESSION_DIR / "AMTEC_Regression_All_Parameters_V3_Filtered_R2.xlsx"

# ============================================================
# MODEL SETTINGS
# ============================================================

TARGET_COL = "fuel_l_per_hr"
POWER_COL  = "power_kw"

GROUP_COLS = [
    "machinery_family",
    "machinery_type"
]

# ALPHA, MIN_RECORDS, MIN_UNIQUE_POWER, MIN_ACCEPTABLE_R2, R2_FILTER_TYPE,
# EXCLUDE_EXTREME_OUTLIERS_FOR_FINAL, and CLIP_NEGATIVE_PREDICTIONS all come from config.

# ============================================================
# LOAD DATA
# ============================================================

df = pd.read_excel(INPUT_FILE, sheet_name=SHEET_NAME)

required_cols = [TARGET_COL, POWER_COL, "machinery_family", "machinery_type"]
missing_cols = [c for c in required_cols if c not in df.columns]

if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

for col in [TARGET_COL, POWER_COL]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

for col in GROUP_COLS:
    df[col] = df[col].astype("string").str.strip()

usable = df.dropna(subset=[TARGET_COL, POWER_COL, "machinery_family", "machinery_type"]).copy()

usable = usable[
    (usable[TARGET_COL] > 0) &
    (usable[TARGET_COL] <= 300) &
    (usable[POWER_COL] > 0) &
    (usable[POWER_COL] <= 500)
].copy()

usable["row_id"] = np.arange(1, len(usable) + 1)

print("Usable records before diagnostics:", len(usable))

# ============================================================
# HELPER FUNCTIONS
# ============================================================

def safe_name(text):
    text = str(text)
    text = re.sub(r"[^A-Za-z0-9]+", "_", text)
    text = re.sub(r"_+", "_", text)
    return text.strip("_")

def build_design_matrix(data, model_form):
    x = data[POWER_COL].astype(float)

    if model_form == "linear":
        X = pd.DataFrame({
            POWER_COL: x
        })

    elif model_form == "quadratic":
        X = pd.DataFrame({
            POWER_COL: x,
            "power_kw_squared": x ** 2
        })

    elif model_form == "log_response":
        X = pd.DataFrame({
            POWER_COL: x
        })

    else:
        raise ValueError("Invalid model_form")

    X = sm.add_constant(X, has_constant="add")
    return X

def get_response(data, model_form):
    y = data[TARGET_COL].astype(float)

    if model_form == "log_response":
        return np.log(y)

    return y

def inverse_transform_prediction(pred, model_form):
    if model_form == "log_response":
        pred = np.exp(pred)

    if CLIP_NEGATIVE_PREDICTIONS:
        pred = np.maximum(pred, 0)

    return pred

def equation_from_model(model, model_form):
    params = model.params.to_dict()

    b0 = params.get("const", 0)
    b1 = params.get(POWER_COL, 0)
    b2 = params.get("power_kw_squared", 0)

    if model_form == "linear":
        return f"Fuel_L_h = {b0:.6f} + {b1:.6f}(Power_kW)"

    if model_form == "quadratic":
        return f"Fuel_L_h = {b0:.6f} + {b1:.6f}(Power_kW) + {b2:.6f}(Power_kW^2)"

    if model_form == "log_response":
        return f"ln(Fuel_L_h) = {b0:.6f} + {b1:.6f}(Power_kW); Fuel_L_h = exp({b0:.6f} + {b1:.6f}(Power_kW))"

    return None

def safe_mape(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    mask = y_true != 0

    if mask.sum() == 0:
        return np.nan

    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

def fit_regression(data, model_scope, group_name, model_form):
    data = data.dropna(subset=[TARGET_COL, POWER_COL]).copy()

    n = len(data)
    unique_power = data[POWER_COL].nunique()

    if n < MIN_RECORDS or unique_power < MIN_UNIQUE_POWER:
        model_summary = pd.DataFrame([{
            "model_scope": model_scope,
            "group_name": group_name,
            "model_form": model_form,
            "model_status": "INSUFFICIENT_DATA",
            "n_observations": n,
            "unique_power_values": unique_power
        }])

        return model_summary, pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    X = build_design_matrix(data, model_form)
    y_model_scale = get_response(data, model_form)
    y_actual = data[TARGET_COL].astype(float)

    try:
        model = sm.OLS(y_model_scale, X).fit()
    except Exception as e:
        model_summary = pd.DataFrame([{
            "model_scope": model_scope,
            "group_name": group_name,
            "model_form": model_form,
            "model_status": f"MODEL_ERROR: {e}",
            "n_observations": n,
            "unique_power_values": unique_power
        }])

        return model_summary, pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    influence = OLSInfluence(model)

    pred_model_scale = model.fittedvalues
    pred_actual_scale = inverse_transform_prediction(pred_model_scale, model_form)

    residual_actual_scale = y_actual - pred_actual_scale

    k = int(model.df_model)
    p = int(model.df_model + 1)

    sst = np.sum((y_actual - y_actual.mean()) ** 2)
    ssres = np.sum(residual_actual_scale ** 2)
    ssr = sst - ssres

    msr = ssr / k if k > 0 else np.nan
    msres = ssres / (n - p) if (n - p) > 0 else np.nan
    f0 = msr / msres if msres and msres != 0 else np.nan
    f_critical = stats.f.ppf(1 - ALPHA, k, n - p) if k > 0 and (n - p) > 0 else np.nan

    leverage_cutoff_lecture = (p + 1) / n
    leverage_cutoff_common_2p = (2 * p) / n
    leverage_cutoff_common_3p = (3 * p) / n
    studentized_cutoff = 3
    cooks_cutoff = 4 / n

    r2_actual = 1 - (ssres / sst) if sst != 0 else np.nan
    adj_r2_actual = 1 - ((n - 1) / (n - p)) * (1 - r2_actual) if (n - p) > 0 else np.nan

    filter_r2_value = adj_r2_actual if R2_FILTER_TYPE == "adjusted" else r2_actual
    passes_r2_filter = pd.notna(filter_r2_value) and filter_r2_value >= MIN_ACCEPTABLE_R2

    model_summary = pd.DataFrame([{
        "model_scope": model_scope,
        "group_name": group_name,
        "model_form": model_form,
        "model_status": "OK",
        "n_observations": n,
        "unique_power_values": unique_power,
        "k_regressors": k,
        "p_parameters_including_intercept": p,
        "df_model": model.df_model,
        "df_residual": model.df_resid,
        "alpha": ALPHA,

        "sst_total_sum_squares": sst,
        "ssr_regression_sum_squares": ssr,
        "ssres_residual_sum_squares": ssres,
        "msr_regression_mean_square": msr,
        "msres_residual_mean_square": msres,

        "f_statistic_actual_scale": f0,
        "f_critical": f_critical,
        "f_p_value_model_scale": model.f_pvalue,
        "f_test_decision": "Reject H0" if pd.notna(f0) and pd.notna(f_critical) and f0 > f_critical else "Fail to reject H0",

        "r_squared_model_scale": model.rsquared,
        "adjusted_r_squared_model_scale": model.rsquared_adj,
        "r_squared_actual_scale": r2_actual,
        "adjusted_r_squared_actual_scale": adj_r2_actual,

        "r2_filter_type": R2_FILTER_TYPE,
        "r2_filter_value": filter_r2_value,
        "min_acceptable_r2": MIN_ACCEPTABLE_R2,
        "passes_r2_filter": passes_r2_filter,
        "r2_filter_decision": "KEEP" if passes_r2_filter else "REMOVE_WEAK_R2",

        "aic": model.aic,
        "bic": model.bic,

        "residual_standard_error_actual_scale": np.sqrt(msres) if pd.notna(msres) else np.nan,
        "mse_actual_scale": np.mean(residual_actual_scale ** 2),
        "rmse_actual_scale": np.sqrt(np.mean(residual_actual_scale ** 2)),
        "mae_actual_scale": np.mean(np.abs(residual_actual_scale)),
        "mape_percent_actual_scale": safe_mape(y_actual, pred_actual_scale),

        "min_power_kw": data[POWER_COL].min(),
        "max_power_kw": data[POWER_COL].max(),
        "avg_power_kw": data[POWER_COL].mean(),
        "min_fuel_l_hr": y_actual.min(),
        "max_fuel_l_hr": y_actual.max(),
        "avg_fuel_l_hr": y_actual.mean(),

        "leverage_cutoff_lecture_p_plus_1_over_n": leverage_cutoff_lecture,
        "leverage_cutoff_common_2p_over_n": leverage_cutoff_common_2p,
        "leverage_cutoff_common_3p_over_n": leverage_cutoff_common_3p,
        "studentized_residual_cutoff": studentized_cutoff,
        "cooks_distance_cutoff_4_over_n": cooks_cutoff,

        "equation": equation_from_model(model, model_form)
    }])

    coef_df = pd.DataFrame({
        "model_scope": model_scope,
        "group_name": group_name,
        "model_form": model_form,
        "parameter": model.params.index,
        "estimate": model.params.values,
        "standard_error": model.bse.values,
        "t_value": model.tvalues.values,
        "p_value": model.pvalues.values,
        "ci_lower_95": model.conf_int(alpha=ALPHA)[0].values,
        "ci_upper_95": model.conf_int(alpha=ALPHA)[1].values,
    })

    coef_df["significant_at_0_05"] = coef_df["p_value"] < ALPHA

    anova_df = pd.DataFrame([
        {
            "model_scope": model_scope,
            "group_name": group_name,
            "model_form": model_form,
            "source": "Regression",
            "df": k,
            "sum_sq": ssr,
            "mean_sq": msr,
            "f_value": f0,
            "p_value_model_scale": model.f_pvalue,
        },
        {
            "model_scope": model_scope,
            "group_name": group_name,
            "model_form": model_form,
            "source": "Residual",
            "df": n - p,
            "sum_sq": ssres,
            "mean_sq": msres,
            "f_value": np.nan,
            "p_value_model_scale": np.nan,
        },
        {
            "model_scope": model_scope,
            "group_name": group_name,
            "model_form": model_form,
            "source": "Total",
            "df": n - 1,
            "sum_sq": sst,
            "mean_sq": np.nan,
            "f_value": np.nan,
            "p_value_model_scale": np.nan,
        }
    ])

    diagnostics = data.copy()

    diagnostics["model_scope"] = model_scope
    diagnostics["group_name"] = group_name
    diagnostics["model_form"] = model_form
    diagnostics["passes_r2_filter"] = passes_r2_filter
    diagnostics["r2_filter_decision"] = "KEEP" if passes_r2_filter else "REMOVE_WEAK_R2"

    diagnostics["actual_fuel_l_hr"] = y_actual
    diagnostics["predicted_fuel_l_hr"] = pred_actual_scale
    diagnostics["residual_actual_scale"] = residual_actual_scale
    diagnostics["absolute_residual"] = np.abs(residual_actual_scale)
    diagnostics["percent_error"] = np.where(
        y_actual != 0,
        diagnostics["absolute_residual"] / y_actual * 100,
        np.nan
    )

    diagnostics["fitted_model_scale"] = pred_model_scale
    diagnostics["residual_model_scale"] = model.resid

    diagnostics["standardized_residual"] = influence.resid_studentized_internal
    diagnostics["studentized_residual"] = influence.resid_studentized_external
    diagnostics["leverage_hi"] = influence.hat_matrix_diag
    diagnostics["cooks_distance"] = influence.cooks_distance[0]
    diagnostics["dffits"] = influence.dffits[0]

    diagnostics["leverage_cutoff_lecture"] = leverage_cutoff_lecture
    diagnostics["leverage_cutoff_2p"] = leverage_cutoff_common_2p
    diagnostics["leverage_cutoff_3p"] = leverage_cutoff_common_3p

    diagnostics["is_leverage_point_lecture"] = diagnostics["leverage_hi"] > leverage_cutoff_lecture
    diagnostics["is_leverage_point_2p"] = diagnostics["leverage_hi"] > leverage_cutoff_common_2p
    diagnostics["is_leverage_point_3p"] = diagnostics["leverage_hi"] > leverage_cutoff_common_3p

    diagnostics["is_possible_outlier_studentized"] = diagnostics["studentized_residual"].abs() > studentized_cutoff
    diagnostics["is_influential_cooks"] = diagnostics["cooks_distance"] > cooks_cutoff

    diagnostics["diagnostic_priority"] = np.select(
        [
            diagnostics["is_possible_outlier_studentized"] & diagnostics["is_influential_cooks"],
            diagnostics["is_possible_outlier_studentized"],
            diagnostics["is_influential_cooks"],
            diagnostics["is_leverage_point_lecture"]
        ],
        [
            "HIGH - outlier and influential",
            "MEDIUM - studentized outlier",
            "MEDIUM - influential Cook's distance",
            "LOW - high leverage"
        ],
        default="NORMAL"
    )

    return model_summary, coef_df, anova_df, diagnostics

def run_all_models(data, label_suffix="FULL"):
    summaries = []
    coefs = []
    anovas = []
    diags = []

    model_forms = ["linear", "quadratic", "log_response"]

    # Global
    for model_form in model_forms:
        s, c, a, d = fit_regression(
            data,
            "GLOBAL",
            f"ALL_MACHINERY_{label_suffix}",
            model_form
        )
        summaries.append(s)
        if not c.empty:
            coefs.append(c)
        if not a.empty:
            anovas.append(a)
        if not d.empty:
            diags.append(d)

    # Family
    for family, group in data.groupby("machinery_family"):
        for model_form in model_forms:
            s, c, a, d = fit_regression(
                group,
                "MACHINERY_FAMILY",
                f"{family}_{label_suffix}",
                model_form
            )
            summaries.append(s)
            if not c.empty:
                coefs.append(c)
            if not a.empty:
                anovas.append(a)
            if not d.empty:
                diags.append(d)

    # Machinery type
    for machine, group in data.groupby("machinery_type"):
        for model_form in model_forms:
            s, c, a, d = fit_regression(
                group,
                "MACHINERY_TYPE",
                f"{machine}_{label_suffix}",
                model_form
            )
            summaries.append(s)
            if not c.empty:
                coefs.append(c)
            if not a.empty:
                anovas.append(a)
            if not d.empty:
                diags.append(d)

    summary_df = pd.concat(summaries, ignore_index=True, sort=False)
    coef_df = pd.concat(coefs, ignore_index=True, sort=False) if coefs else pd.DataFrame()
    anova_df = pd.concat(anovas, ignore_index=True, sort=False) if anovas else pd.DataFrame()
    diag_df = pd.concat(diags, ignore_index=True, sort=False) if diags else pd.DataFrame()

    return summary_df, coef_df, anova_df, diag_df

# ============================================================
# FIRST PASS: FULL DATA
# ============================================================

model_summary_full, coef_full, anova_full, diagnostics_full = run_all_models(
    usable,
    label_suffix="FULL"
)

global_diag_linear = diagnostics_full[
    (diagnostics_full["model_scope"] == "GLOBAL") &
    (diagnostics_full["group_name"] == "ALL_MACHINERY_FULL") &
    (diagnostics_full["model_form"] == "linear")
].copy()

extreme_row_ids = set(
    global_diag_linear[
        global_diag_linear["is_possible_outlier_studentized"] == True
    ]["row_id"].tolist()
)

if EXCLUDE_EXTREME_OUTLIERS_FOR_FINAL:
    final_data = usable[~usable["row_id"].isin(extreme_row_ids)].copy()
else:
    final_data = usable.copy()

print("Extreme global outliers identified:", len(extreme_row_ids))
print("Records for final models:", len(final_data))

# ============================================================
# SECOND PASS: FINAL DATA
# ============================================================

model_summary_final, coef_final, anova_final, diagnostics_final = run_all_models(
    final_data,
    label_suffix="FINAL"
)

# ============================================================
# FILTER WEAK R2 MODELS
# ============================================================

ok_models = model_summary_final[
    model_summary_final["model_status"] == "OK"
].copy()

strong_models = ok_models[
    ok_models["passes_r2_filter"] == True
].copy()

weak_models_removed = ok_models[
    ok_models["passes_r2_filter"] == False
].copy()

# Best strong models only
best_strong_models = (
    strong_models
    .sort_values(
        [
            "model_scope",
            "group_name",
            "rmse_actual_scale",
            "aic",
            "adjusted_r_squared_actual_scale"
        ],
        ascending=[True, True, True, True, False]
    )
    .groupby(["model_scope", "group_name"], as_index=False)
    .first()
)

prediction_hierarchy_filtered = best_strong_models[
    [
        "model_scope",
        "group_name",
        "model_form",
        "model_status",
        "n_observations",
        "unique_power_values",
        "equation",
        "min_power_kw",
        "max_power_kw",
        "avg_power_kw",
        "avg_fuel_l_hr",
        "r_squared_actual_scale",
        "adjusted_r_squared_actual_scale",
        "rmse_actual_scale",
        "mae_actual_scale",
        "mape_percent_actual_scale",
        "aic",
        "bic",
        "r2_filter_type",
        "r2_filter_value",
        "min_acceptable_r2",
        "r2_filter_decision"
    ]
].copy()

# ============================================================
# FALLBACK HIERARCHY NOTES
# ============================================================

fallback_notes = []

all_machine_types = sorted(final_data["machinery_type"].dropna().unique())

for machine in all_machine_types:
    machine_family = final_data.loc[
        final_data["machinery_type"] == machine,
        "machinery_family"
    ].dropna().iloc[0]

    machine_model = prediction_hierarchy_filtered[
        (prediction_hierarchy_filtered["model_scope"] == "MACHINERY_TYPE") &
        (prediction_hierarchy_filtered["group_name"] == f"{machine}_FINAL")
    ]

    family_model = prediction_hierarchy_filtered[
        (prediction_hierarchy_filtered["model_scope"] == "MACHINERY_FAMILY") &
        (prediction_hierarchy_filtered["group_name"] == f"{machine_family}_FINAL")
    ]

    global_model = prediction_hierarchy_filtered[
        (prediction_hierarchy_filtered["model_scope"] == "GLOBAL") &
        (prediction_hierarchy_filtered["group_name"] == "ALL_MACHINERY_FINAL")
    ]

    if len(machine_model) > 0:
        selected_level = "MACHINERY_TYPE"
        selected_group = f"{machine}_FINAL"
    elif len(family_model) > 0:
        selected_level = "MACHINERY_FAMILY"
        selected_group = f"{machine_family}_FINAL"
    elif len(global_model) > 0:
        selected_level = "GLOBAL"
        selected_group = "ALL_MACHINERY_FINAL"
    else:
        selected_level = "NO_ACCEPTABLE_MODEL"
        selected_group = None

    fallback_notes.append({
        "machinery_type": machine,
        "machinery_family": machine_family,
        "recommended_prediction_level": selected_level,
        "recommended_prediction_group": selected_group,
        "min_acceptable_r2": MIN_ACCEPTABLE_R2,
        "r2_filter_type": R2_FILTER_TYPE
    })

fallback_hierarchy = pd.DataFrame(fallback_notes)

# ============================================================
# DIAGNOSTIC EXTRACTS
# ============================================================

diagnostics_final_filtered = diagnostics_final[
    diagnostics_final["passes_r2_filter"] == True
].copy()

high_leverage = diagnostics_final[
    diagnostics_final["is_leverage_point_lecture"] == True
].copy()

studentized_outliers = diagnostics_final[
    diagnostics_final["is_possible_outlier_studentized"] == True
].copy()

influential_points = diagnostics_final[
    diagnostics_final["is_influential_cooks"] == True
].copy()

priority_review = diagnostics_final[
    diagnostics_final["diagnostic_priority"] != "NORMAL"
].copy()

# ============================================================
# SAVE OUTPUT
# ============================================================

with pd.ExcelWriter(OUTPUT_EXCEL, engine="openpyxl") as writer:
    usable.to_excel(writer, sheet_name="Original Regression Dataset", index=False)
    final_data.to_excel(writer, sheet_name="Final Regression Dataset", index=False)

    model_summary_full.to_excel(writer, sheet_name="Model Parameters FULL", index=False)
    model_summary_final.to_excel(writer, sheet_name="Model Parameters FINAL", index=False)

    coef_final.to_excel(writer, sheet_name="Coefficient Parameters FINAL", index=False)
    anova_final.to_excel(writer, sheet_name="ANOVA FINAL", index=False)

    prediction_hierarchy_filtered.to_excel(writer, sheet_name="Prediction Hierarchy FILTERED", index=False)
    fallback_hierarchy.to_excel(writer, sheet_name="Fallback Hierarchy", index=False)

    strong_models.to_excel(writer, sheet_name="Strong Models Kept", index=False)
    weak_models_removed.to_excel(writer, sheet_name="Weak Models Removed", index=False)

    diagnostics_final.to_excel(writer, sheet_name="Residual Diagnostics FINAL", index=False)
    diagnostics_final_filtered.to_excel(writer, sheet_name="Diagnostics Strong Models", index=False)

    priority_review.to_excel(writer, sheet_name="Priority Review Points", index=False)
    high_leverage.to_excel(writer, sheet_name="High Leverage Points", index=False)
    studentized_outliers.to_excel(writer, sheet_name="Studentized Outliers", index=False)
    influential_points.to_excel(writer, sheet_name="Influential Points", index=False)

print("Saved:", OUTPUT_EXCEL)

# ============================================================
# PLOTS FOR STRONG MODELS ONLY
# ============================================================

def plot_diagnostics_for_model(diag, title_prefix, output_prefix):
    if len(diag) < 5:
        return

    plt.figure(figsize=(8, 6))
    plt.scatter(diag["predicted_fuel_l_hr"], diag["residual_actual_scale"], alpha=0.7)
    plt.axhline(0, linestyle="--")
    plt.xlabel("Fitted Values")
    plt.ylabel("Residuals")
    plt.title(f"Residuals vs Fitted - {title_prefix}")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"{output_prefix}_residuals_vs_fitted.png", dpi=300)
    plt.close()

    plt.figure(figsize=(8, 6))
    sm.qqplot(diag["residual_actual_scale"], line="45", fit=True)
    plt.title(f"Normal Q-Q Plot - {title_prefix}")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"{output_prefix}_qqplot.png", dpi=300)
    plt.close()

    plt.figure(figsize=(8, 6))
    plt.scatter(range(len(diag)), diag["studentized_residual"], alpha=0.7)
    plt.axhline(3, linestyle="--")
    plt.axhline(-3, linestyle="--")
    plt.xlabel("Observation Index")
    plt.ylabel("Studentized Residual")
    plt.title(f"Studentized Residuals - {title_prefix}")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"{output_prefix}_studentized_residuals.png", dpi=300)
    plt.close()

    plt.figure(figsize=(8, 6))
    plt.scatter(range(len(diag)), diag["leverage_hi"], alpha=0.7)
    if "leverage_cutoff_lecture" in diag.columns:
        plt.axhline(diag["leverage_cutoff_lecture"].iloc[0], linestyle="--")
    plt.xlabel("Observation Index")
    plt.ylabel("Leverage hᵢ")
    plt.title(f"Leverage Values - {title_prefix}")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"{output_prefix}_leverage.png", dpi=300)
    plt.close()

for _, model_row in prediction_hierarchy_filtered.iterrows():
    scope = model_row["model_scope"]
    group = model_row["group_name"]
    form = model_row["model_form"]

    diag = diagnostics_final[
        (diagnostics_final["model_scope"] == scope) &
        (diagnostics_final["group_name"] == group) &
        (diagnostics_final["model_form"] == form)
    ]

    plot_diagnostics_for_model(
        diag,
        f"{scope} - {group} - {form}",
        f"{safe_name(scope)}_{safe_name(group)}_{safe_name(form)}"
    )

# ============================================================
# SUMMARY
# ============================================================

summary_txt = OUTPUT_DIR / "AMTEC_Regression_V3_Filtered_R2_Summary.txt"

summary_text = f"""
AMTEC REGRESSION PARAMETERS V3 FILTERED R2 SUMMARY

Original usable records: {len(usable)}
Extreme global outliers identified: {len(extreme_row_ids)}
Final regression records: {len(final_data)}

Minimum acceptable R2: {MIN_ACCEPTABLE_R2}
R2 filter type: {R2_FILTER_TYPE}

Strong models kept: {len(strong_models)}
Weak models removed: {len(weak_models_removed)}
Filtered prediction hierarchy models: {len(prediction_hierarchy_filtered)}

Outlier exclusion enabled: {EXCLUDE_EXTREME_OUTLIERS_FOR_FINAL}

Priority review points: {len(priority_review)}
Studentized outliers: {len(studentized_outliers)}
High leverage points: {len(high_leverage)}
Influential Cook's distance points: {len(influential_points)}

Output Excel:
{OUTPUT_EXCEL}

Output folder:
{OUTPUT_DIR}
"""

with open(summary_txt, "w", encoding="utf-8") as f:
    f.write(summary_text)

print(summary_text)
print("DONE.")

## 6. Results Summary - Data Pre-processing

In [ ]:
print("=" * 60)
print("PIPELINE COMPLETE — Output locations")
print("=" * 60)
print()
print("ABEMIS Inventory")
print("  Extracted:      ", ABEMIS_EXTRACTED_DIR)
print("  Diagnostics:    ", ABEMIS_DIAG_DIR)
print("  Fuel classes:   ", ABEMIS_FUEL_DIR)
print()
print("AMTEC Test Reports")
print("  Extracted:      ", AMTEC_EXTRACTION_DIR)
print("  Analytics:      ", AMTEC_ANALYTICS_DIR)
print("  Regression V3:  ", AMTEC_REGRESSION_DIR)

# Machine Learning Implementation

Hierarchical OLS is interpretable but assumes monotonic, parametric relationships.

**Limitations of linear models:**

* Fuel consumption is empirically non-monotonic above rated-power saturation
* Interacts with categorical features (machinery family, deployment context)

**Random Forest advantages:**

* Relaxes monotonicity and linearity assumptions
* Keeps inference computationally cheap
* SHAP and LIME provide post-hoc interpretability

**Three artifacts produced:**

1. OLS baseline (summary)
2. RF model
3. Per-feature global + local explanations

## Setup: Baseline Model

Hierarchical OLS regression applied to 376-record *Clean Record Level* sheet.

**Model hierarchy:**

$$\text{Model selection: GLOBAL} \to \text{MACHINERY\_FAMILY} \to \text{MACHINERY\_TYPE}$$

**Evaluation per scope:**

* Linear form: $\hat{y} = \beta_0 + \beta_1 x$
* Quadratic form: $\hat{y} = \beta_0 + \beta_1 x + \beta_2 x^2$
* Log-response form: $\ln(y) = \beta_0 + \beta_1 x$

**Model filtering:**

* Discard models with adjusted $R^2 < 0.50$
* Best surviving model at each level forms fallback hierarchy for prediction

In [ ]:
# OLS V3 has already been run inline by the earlier 'AMTEC Regression Modeling' cell.
# Here we just load the persisted fallback hierarchy and display it.

import pandas as pd

reg_xlsx = AMTEC_REGRESSION_DIR / 'AMTEC_Regression_All_Parameters_V3_Filtered_R2.xlsx'
if not reg_xlsx.exists():
    raise FileNotFoundError(
        f"Could not find {reg_xlsx}. Run the 'AMTEC Regression Modeling' cell first.")

fallback = pd.read_excel(reg_xlsx, sheet_name='Fallback Hierarchy')
print('Fallback Hierarchy (one row per machinery_type):')
print(fallback[['machinery_type', 'recommended_prediction_level']].to_string(index=False))


### 1A. OLS Coefficient Table

**First sanity checks:**

* Does sign of `power_kw` come out positive? (more power → more fuel)
* Are t-values large enough to reject null hypothesis?
* Is p-value statistically significant?

**Why filter to GLOBAL/linear first?**

* Simplest interpretable form
* Every other scope or response form is refinement on this baseline

**What to look for:**

* Coefficient sign correctness
* t-statistics magnitude
* p-values for significance
* Confidence intervals

In [ ]:
import pandas as pd
from IPython.display import display

reg_xlsx = AMTEC_REGRESSION_DIR / 'AMTEC_Regression_All_Parameters_V3_Filtered_R2.xlsx'
coef_df = pd.read_excel(reg_xlsx, sheet_name='Coefficient Parameters FINAL')

# Filter to GLOBAL scope, linear form -- the simplest interpretable baseline.
global_linear_coef = coef_df[
    (coef_df['model_scope'] == 'GLOBAL') &
    (coef_df['group_name']  == 'ALL_MACHINERY_FINAL') &
    (coef_df['model_form']  == 'linear')
].reset_index(drop=True)

display_cols = ['term', 'coef', 'std_err', 't_value', 'p_value',
                'r_squared_actual_scale', 'adjusted_r_squared_actual_scale']
available = [c for c in display_cols if c in global_linear_coef.columns]
print('GLOBAL / linear -- OLS coefficients')
display(global_linear_coef[available])


### 1B. OLS Diagnostic Plots

**Residual vs. Fitted plot:**

* Reveals heteroscedasticity (variance grows with predicted fuel)
* Incorrect heteroscedasticity → standard errors wrong → confidence intervals wrong

**Q-Q plot (Quantile-Quantile):**

* Reveals tail-heaviness or skewness in residuals
* Violates normality assumption that OLS p-values rely on

**Why both plots required:**

* IE 273 reporting convention (Philippine agricultural engineering standard)
* Validates whether OLS p-values can be trusted
* Without diagnostic validation, significance claims are unreliable

In [ ]:
from IPython.display import Image, display

plots = [
    AMTEC_REGRESSION_DIR / 'GLOBAL_ALL_MACHINERY_FINAL_quadratic_residuals_vs_fitted.png',
    AMTEC_REGRESSION_DIR / 'GLOBAL_ALL_MACHINERY_FINAL_quadratic_qqplot.png',
]
for p in plots:
    if p.exists():
        print(p.name)
        display(Image(str(p), width=700))
    else:
        print(f'[not found] {p.name}')


## Setup: Random Forest Regressor

**Why Random Forest as the ML choice?**

1. **Mixed feature types**: Bagged trees handle numeric + categorical features without manual interaction-term engineering

2. **Robustness to outliers**: Mis-transcribed decimals in AMTEC PDFs don't dominate the fit like they would in OLS

3. **Interpretability**: Tree-based SHAP is exact and computationally cheap on forests
   - Interpretability layer essentially free

**Hyperparameters:**

* Scikit-learn defaults (100 estimators, no max depth)
* For this dataset with n=376 training records:
  - Sufficient to capture complex interactions
  - Low risk of overfitting given holdout CV results
  
$$\text{RF prediction} = \frac{1}{B} \sum_{b=1}^{B} T_b(\mathbf{x})$$

where $B$ = number of trees, $T_b$ = individual tree prediction

In [ ]:
# ============================================================
# Random Forest fuel-consumption regressor (inlined from models/random_forest.py)
# ============================================================
import pickle
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, train_test_split

INPUT_FILE   = AMTEC_ANALYTICS_DIR  / "AMTEC_Test_Report_Fuel_Power_Analytics_V2.xlsx"
MODEL_FILE   = AMTEC_REGRESSION_DIR / "rf_model.pkl"
ENCODER_FILE = AMTEC_REGRESSION_DIR / "rf_encoders.pkl"
OUTPUT_EXCEL = AMTEC_REGRESSION_DIR / "RF_Predictions.xlsx"

# ---- load & preprocess ----------------------------------------------------
def rf_load_data():
    df = pd.read_excel(INPUT_FILE, sheet_name="Clean Record Level")
    required = [TARGET_COL] + NUMERIC_FEATURES + CATEGORICAL_FEATURES
    df = df.dropna(subset=required).copy()
    for c in [TARGET_COL] + NUMERIC_FEATURES:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df = df.dropna(subset=[TARGET_COL] + NUMERIC_FEATURES)

    encoders = {}
    for c in CATEGORICAL_FEATURES:
        cats = sorted(df[c].astype(str).unique())
        mapping = {v: i for i, v in enumerate(cats)}
        df[f"{c}_code"] = df[c].astype(str).map(mapping).astype(int)
        encoders[c] = mapping
    return df, encoders

# ---- train ----------------------------------------------------------------
def rf_train(df):
    X = df[FEATURE_COLS].values
    y = df[TARGET_COL].values
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE)
    model = RandomForestRegressor(
        n_estimators=300, max_depth=None,
        min_samples_split=5, min_samples_leaf=2,
        random_state=RANDOM_STATE, n_jobs=-1)
    model.fit(X_train, y_train)
    return model, X_train, X_test, y_train, y_test

# ---- evaluate -------------------------------------------------------------
def rf_eval(model, X, y):
    pred = model.predict(X)
    return {"r2": r2_score(y, pred),
            "mae": mean_absolute_error(y, pred),
            "rmse": float(np.sqrt(mean_squared_error(y, pred)))}

# ---- 5-fold cross-validation ---------------------------------------------
def rf_cv(df, n_splits=5):
    X, y = df[FEATURE_COLS].values, df[TARGET_COL].values
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    rows = []
    for fold, (tr, te) in enumerate(kf.split(X), 1):
        m = RandomForestRegressor(
            n_estimators=300, max_depth=None,
            min_samples_split=5, min_samples_leaf=2,
            random_state=RANDOM_STATE, n_jobs=-1).fit(X[tr], y[tr])
        p = m.predict(X[te])
        rows.append({"fold": fold, "n_train": len(tr), "n_test": len(te),
                     "r2": r2_score(y[te], p),
                     "mae": mean_absolute_error(y[te], p),
                     "rmse": float(np.sqrt(mean_squared_error(y[te], p)))})
    folds = pd.DataFrame(rows)
    summary = pd.DataFrame([
        {"fold": "mean", **{c: folds[c].mean() for c in ["n_train","n_test","r2","mae","rmse"]}},
        {"fold": "std",  **{c: folds[c].std()  for c in ["n_train","n_test","r2","mae","rmse"]}},
    ])
    return pd.concat([folds, summary], ignore_index=True)

# ---- run ------------------------------------------------------------------
df, encoders = rf_load_data()
print(f"Records after preprocessing: {len(df)}")
print(f"Features ({len(FEATURE_COLS)}): {FEATURE_COLS}")

rf_model, X_train, X_test, y_train, y_test = rf_train(df)
train_m = rf_eval(rf_model, X_train, y_train)
test_m  = rf_eval(rf_model, X_test,  y_test)
cv_df   = rf_cv(df, n_splits=5)

print("\nRandom Forest holdout metrics:")
for label, m in [("train", train_m), ("test", test_m)]:
    print(f"  {label}: R2={m['r2']:.4f}  MAE={m['mae']:.4f} L/h  RMSE={m['rmse']:.4f} L/h")
cv_mean = float(cv_df.loc[cv_df['fold']=='mean','r2'].iloc[0])
cv_std  = float(cv_df.loc[cv_df['fold']=='std', 'r2'].iloc[0])
print(f"  cv (5-fold mean): R2={cv_mean:.4f}  (std={cv_std:.4f})")

# ---- persist model + predictions xlsx ------------------------------------
with open(MODEL_FILE,   "wb") as f: pickle.dump(rf_model, f)
with open(ENCODER_FILE, "wb") as f: pickle.dump({"encoders": encoders, "feature_cols": FEATURE_COLS}, f)
print(f"\nModel saved   : {MODEL_FILE}")
print(f"Encoders saved: {ENCODER_FILE}")

df["rf_predicted_fuel_l_hr"] = rf_model.predict(df[FEATURE_COLS].values)
if CLIP_NEGATIVE_PREDICTIONS:
    df["rf_predicted_fuel_l_hr"] = df["rf_predicted_fuel_l_hr"].clip(lower=0)
df["rf_residual"]     = df[TARGET_COL] - df["rf_predicted_fuel_l_hr"]
df["rf_abs_residual"] = df["rf_residual"].abs()
df["rf_pct_error"]    = np.where(df[TARGET_COL] != 0,
                                 df["rf_abs_residual"] / df[TARGET_COL] * 100, np.nan)

importances = pd.DataFrame({
    "feature": FEATURE_COLS, "importance": rf_model.feature_importances_,
}).sort_values("importance", ascending=False)

metrics_df = pd.DataFrame([
    {"split": "train", **train_m, "n": len(y_train)},
    {"split": "test",  **test_m,  "n": len(y_test)},
    {"split": "all",   **rf_eval(rf_model, df[FEATURE_COLS].values, df[TARGET_COL].values), "n": len(df)},
])

with pd.ExcelWriter(OUTPUT_EXCEL, engine="openpyxl") as w:
    df.to_excel(w,          sheet_name="RF Predictions",    index=False)
    metrics_df.to_excel(w,  sheet_name="Metrics",           index=False)
    cv_df.to_excel(w,       sheet_name="CV Metrics",        index=False)
    importances.to_excel(w, sheet_name="Feature Importance",index=False)
    for col, mapping in encoders.items():
        pd.DataFrame([{"category": k, "code": v} for k,v in mapping.items()]).to_excel(
            w, sheet_name=f"Encoder {col}"[:31], index=False)
print(f"Predictions saved: {OUTPUT_EXCEL}")


### 2A. SHAP Feature Importance

**Why SHAP?** Random Forest predictions are opaque per-record (no closed-form expression for "why did the model predict 4.2 L/h?").

SHAP (SHapley Additive exPlanations) gives every feature a contribution that sums to exactly the prediction.

**Two key artifacts:**

* **Bar plot (mean |SHAP|)**: Global feature ranking
  - Answers "which features drive the model overall?"
  
* **Beeswarm plot**: Per-record dots colored by feature value
  - Answers "does high power push predictions up or down?"

**Why Tree-based SHAP?**

* TreeExplainer is exact and computationally cheap on forests
* Interpretability layer comes essentially for free

In [ ]:
# ============================================================
# SHAP feature importance for the trained Random Forest (inlined)
# ============================================================
import numpy as np
import pandas as pd
import shap
import matplotlib.pyplot as plt
from IPython.display import Image, display

# rf_model, df, encoders, FEATURE_COLS all come from the Random Forest cell above.
X_df = df[FEATURE_COLS]

explainer   = shap.TreeExplainer(rf_model)
shap_values = explainer.shap_values(X_df)

mean_abs_shap = np.abs(shap_values).mean(axis=0)
importance_df = pd.DataFrame({
    "feature": FEATURE_COLS,
    "mean_abs_shap": mean_abs_shap,
}).sort_values("mean_abs_shap", ascending=False)

# Save plots
shap_bar_path = AMTEC_REGRESSION_DIR / "shap_summary_bar.png"
plt.figure()
shap.summary_plot(shap_values, X_df, plot_type="bar", show=False)
plt.tight_layout(); plt.savefig(shap_bar_path, dpi=300, bbox_inches="tight"); plt.close()

shap_beeswarm_path = AMTEC_REGRESSION_DIR / "shap_summary_beeswarm.png"
plt.figure()
shap.summary_plot(shap_values, X_df, show=False)
plt.tight_layout(); plt.savefig(shap_beeswarm_path, dpi=300, bbox_inches="tight"); plt.close()

# Save xlsx with SHAP values per record
shap_xlsx = AMTEC_REGRESSION_DIR / "SHAP_Analysis.xlsx"
shap_df = pd.DataFrame(shap_values, columns=[f"shap_{c}" for c in FEATURE_COLS])
output_df = pd.concat([df.reset_index(drop=True), shap_df], axis=1)
with pd.ExcelWriter(shap_xlsx, engine="openpyxl") as w:
    importance_df.to_excel(w, sheet_name="Feature Importance",      index=False)
    output_df.to_excel(w,     sheet_name="SHAP Values Per Record",  index=False)

print('Top SHAP feature importances (mean |SHAP|):')
display(importance_df.reset_index(drop=True))
for p in [shap_bar_path, shap_beeswarm_path]:
    if p.exists():
        print(p.name)
        display(Image(str(p), width=750))


### 2B. LIME Local Explanations

**Why LIME on top of SHAP?**

* SHAP: Model-consistent explanation derived from tree structure
* LIME: Independent linear surrogate around each prediction
* Different theoretical grounds

**Why both matter:**

* Agreement between methods is stronger evidence model uses features correctly
* LIME easier to communicate to non-ML stakeholders:
  - "For this record, +X L/h came from high power_kw, -Y from year"
  - Local surrogate is interpretable algebraic expression

**This study applies LIME to 10 representative holdout records** for validation and stakeholder communication.

In [ ]:
# ============================================================
# LIME local explanations for sampled predictions (inlined)
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from lime import lime_tabular
from IPython.display import Image, display

N_SAMPLES = 10
lime_xlsx = AMTEC_REGRESSION_DIR / "LIME_Explanations.xlsx"

# Reuse rf_model, df, encoders, FEATURE_COLS, TARGET_COL, CATEGORICAL_FEATURES from earlier cells.
X_train = df[FEATURE_COLS].values

cat_indices = [FEATURE_COLS.index(f"{c}_code") for c in CATEGORICAL_FEATURES]
cat_names = {
    FEATURE_COLS.index(f"{c}_code"): list(encoders[c].keys())
    for c in CATEGORICAL_FEATURES
}

explainer = lime_tabular.LimeTabularExplainer(
    training_data=X_train,
    feature_names=FEATURE_COLS,
    mode="regression",
    categorical_features=cat_indices,
    categorical_names=cat_names,
    random_state=42,
)

rng = np.random.default_rng(42)
sample_idx = rng.choice(len(df), size=min(N_SAMPLES, len(df)), replace=False)
all_explanations = []

for i, idx in enumerate(sample_idx):
    instance  = X_train[idx]
    exp       = explainer.explain_instance(instance, rf_model.predict, num_features=len(FEATURE_COLS))
    actual    = float(df.iloc[idx][TARGET_COL])
    predicted = float(rf_model.predict([instance])[0])

    row = {
        "sample_index":        int(idx),
        "machinery_type":      df.iloc[idx]["machinery_type"],
        "power_kw":            float(df.iloc[idx]["power_kw"]),
        "actual_fuel_l_hr":    actual,
        "predicted_fuel_l_hr": predicted,
    }
    for feat, weight in exp.as_list():
        row[f"lime_{feat}"] = weight
    all_explanations.append(row)

    fig = exp.as_pyplot_figure()
    fig.suptitle(f"LIME - Sample {idx}  |  {df.iloc[idx]['machinery_type']}  |  "
                 f"Actual={actual:.3f}  Pred={predicted:.3f}", fontsize=9)
    fig.tight_layout()
    fig.savefig(AMTEC_REGRESSION_DIR / f"lime_example_{i}.png", dpi=150, bbox_inches="tight")
    plt.close(fig)

explanations_df = pd.DataFrame(all_explanations)
with pd.ExcelWriter(lime_xlsx, engine="openpyxl") as w:
    explanations_df.to_excel(w, sheet_name="LIME Explanations", index=False)

print(f"LIME explanations saved to: {lime_xlsx}")
p = AMTEC_REGRESSION_DIR / "lime_example_0.png"
if p.exists():
    print('lime_example_0.png')
    display(Image(str(p), width=750))


## Improvements

### ABEMIS Context Features

Three new numeric columns derived from the ABEMIS inventory were added to the Random Forest feature set:

| Feature | Description |
|---|---|
| `abemis_total_count` | Total ABEMIS units matched to this machinery type |
| `abemis_region_breadth` | Number of distinct regions where the type is deployed |
| `abemis_dominant_region_share` | Share of units in the most common single region |

These features provide demand-side context from the ABEMIS dataset, complementing the supply-side AMTEC performance data. SHAP analysis confirms a combined contribution of ~9% of mean absolute impact.

### OCR Retraining

The AMTEC extractor (`data/ingestion/amtec_pdf_extractor.py`) was extended with OCR fallback via `pytesseract` + `pdf2image` when PyMuPDF text extraction yields fewer than `MIN_TEXT_LENGTH` characters. A pass over 267 low-text/scanned PDFs successfully OCR-extracted 266 and recovered an additional **24 model-usable records**, bringing the clean training corpus from 371 to **376** records.

## Results Summary - Machine Learning Implementation

| Metric | OLS Global | OLS Hierarchical | Random Forest |
|--------|-----------|-----------------|---------------|
| **R²** | 0.704 | 0.703 | **0.916** |
| **MAE (L/h)** | 1.882 | 1.879 | **0.950** |
| **RMSE (L/h)** | 2.875 | 2.879 | **1.534** |
| Training records | 300 | 300 | 300 (80%) |
| Test records | 76 | 76 | 76 (20%) |

**Key findings:**

* OLS metrics are out-of-sample on same 76-record holdout as RF
* Hierarchical OLS applies per-type fallback (type → family → global)
* Results land within rounding of global linear form
* **Random Forest substantially outperforms both OLS variants**

## Analysis of Results

**Dominant driver: Rated power** ($\text{power\_kw}$)

* SHAP global importance: ~76% of mean absolute impact (mean |SHAP| ~3.63 L/h)
* Aligns with thermodynamic theory: fuel consumption scales quasi-linearly with engine load
* Load bounded by rated power output

**Secondary drivers:**

* **Machinery context features** (~3.5% combined)
  - `abemis_total_count`: Total ABEMIS units
  - `abemis_region_breadth`: Geographic spread
  - `abemis_dominant_region_share`: Concentration
  - Suggests machinery with higher national inventory have better utilization

* **Categorical features** (machinery type, family, fuel type)
  - Capture machinery-specific efficiency differences

**Fuel intensity relationship:**

$$\text{Fuel} \propto \text{Power} + \text{interactions}$$

Random Forest recovers non-linear interactions that linear regression misses.

# Conclusion

**Model Performance:**

* Random Forest outperforms OLS across all metrics:
  - R² 0.916 vs 0.704
  - RMSE 1.534 vs 2.875 L/h
  - MAE 0.950 vs 1.882 L/h
* Confirms ensemble methods better capture non-linear, multi-category structure of AMTEC data

**Interpretability:**

* SHAP and LIME independently confirm rated power as primary driver
* SHAP provides globally consistent rankings
* LIME confirms same ordering at individual record level
* Increased confidence in model's internal logic

**Tractability:**

* Both explainability methods strengthen traceability
* Non-experts can understand per-record predictions
* Results can be communicated to Philippine agricultural engineers and policy makers
* Framework enables principled validation against field data

# Recommendations

**1. Integrate RAED cropping calendar (planned)**

* Add seasonal demand layer: `data/raed_cropping_calendar.py`
* New features: `is_in_season`, `crop_stage_intensity`
* Enable region-month demand scoring via `analysis/abemis_fuel_scoring.py`
* Constitutes B2 scoring module

**2. Continue OCR pass on additional PDF sources**

* Current 376 clean records (post-OCR consolidation) are subset of available AMTEC reports
* Running extractor against additional PDF batches expands training coverage
* Especially valuable for underrepresented machinery types (small-scale tillers, post-harvest equipment)

**3. Expand ABEMIS to AMTEC type mapping**

* Recover unmapped records
* Reduce reliance on proxy assignments

**4. In-field telemetry validation**

* Validate predictions against actual fuel consumption measurements
* Pilot deployment of GPS + flow-sensor-equipped machinery
* Assess real-world accuracy beyond standardized AMTEC test conditions